In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "c49f178d-7e56-43ae-a5c9-61e46605bea5",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "✅ Siap merekam data...\n",
      "⚡ KICKSTART! Memaksa motor jalan dulu...\n",
      "⚡ Stabilizing... Masuk mode aman.\n",
      "Rec: t=5.0s | PWM=145 | RPM=1080 [OK]\n",
      "Rec: t=9.98s | PWM=122 | RPM=720 [OK]\n",
      "Rec: t=15.05s | PWM=141 | RPM=0 [MACET (Bantu putar pakai tangan dikit)]\n",
      "Rec: t=20.03s | PWM=122 | RPM=0 [MACET (Bantu putar pakai tangan dikit)]\n",
      "Rec: t=25.0s | PWM=158 | RPM=3120 [OK]\n",
      "Rec: t=29.98s | PWM=163 | RPM=5400 [OK]\n",
      "Rec: t=34.98s | PWM=156 | RPM=5040 [OK]\n",
      "Rec: t=39.96s | PWM=152 | RPM=3840 [OK]\n",
      "Rec: t=45.03s | PWM=170 | RPM=5400 [OK]\n",
      "Rec: t=50.02s | PWM=145 | RPM=4920 [OK]\n",
      "Rec: t=55.01s | PWM=143 | RPM=2280 [OK]\n",
      "Rec: t=59.99s | PWM=144 | RPM=2760 [OK]\n",
      "Rec: t=64.97s | PWM=122 | RPM=0 [MACET (Bantu putar pakai tangan dikit)]\n",
      "Rec: t=70.04s | PWM=138 | RPM=1920 [OK]\n",
      "Rec: t=75.03s | PWM=166 | RPM=6360 [OK]\n",
      "Rec: t=80.02s | PWM=166 | RPM=6600 [OK]\n",
      "Rec: t=85.01s | PWM=165 | RPM=4800 [OK]\n",
      "Rec: t=89.97s | PWM=160 | RPM=5160 [OK]\n",
      "Rec: t=95.04s | PWM=165 | RPM=4200 [OK]\n",
      "Rec: t=100.04s | PWM=166 | RPM=4800 [OK]\n",
      "Rec: t=105.03s | PWM=170 | RPM=7200 [OK]\n",
      "Rec: t=110.0s | PWM=161 | RPM=4560 [OK]\n",
      "Rec: t=114.99s | PWM=153 | RPM=4920 [OK]\n",
      "Rec: t=119.97s | PWM=141 | RPM=3480 [OK]\n",
      "\n",
      "✅ SUKSES! Data tersimpan (240 baris).\n"
     ]
    }
   ],
   "source": [
    "import serial\n",
    "import time\n",
    "import random\n",
    "import pandas as pd\n",
    "\n",
    "# --- KONFIGURASI ---\n",
    "PORT = 'COM9'   # <--- Pastikan Port Benar\n",
    "BAUDRATE = 115200\n",
    "DURASI_REKAM = 120  # 2 Menit\n",
    "\n",
    "# --- SETTING BARU (TENAGA PAS) ---\n",
    "MAX_PWM = 170   # Batas Atas (Cukup kencang tapi gak bahaya)\n",
    "MIN_PWM = 120   # Batas Bawah (Biar gak mati/berhenti)\n",
    "STEP_UBAH = 5   # Perubahan gas (biar agak responsif dikit)\n",
    "\n",
    "# Setup Serial\n",
    "try:\n",
    "    ser = serial.Serial(PORT, BAUDRATE, timeout=1)\n",
    "    time.sleep(2)\n",
    "    ser.reset_input_buffer()\n",
    "    print(\"✅ Siap merekam data...\")\n",
    "except:\n",
    "    print(\"❌ Gagal koneksi serial.\")\n",
    "    ser = None\n",
    "\n",
    "data_log = []\n",
    "start_time = time.time()\n",
    "current_pwm = 130 \n",
    "\n",
    "if ser:\n",
    "    try:\n",
    "        # --- 1. KICKSTART (TENDANGAN AWAL) ---\n",
    "        print(\"⚡ KICKSTART! Memaksa motor jalan dulu...\")\n",
    "        ser.write(b\"200\\n\")  # Tembak tenaga besar sebentar\n",
    "        time.sleep(0.5)      # Tahan setengah detik\n",
    "        print(\"⚡ Stabilizing... Masuk mode aman.\")\n",
    "        \n",
    "        # --- 2. MULAI REKAM ---\n",
    "        while (time.time() - start_time) < DURASI_REKAM:\n",
    "            \n",
    "            # Random Walk (Naik turun halus)\n",
    "            change = random.randint(-STEP_UBAH, STEP_UBAH)\n",
    "            current_pwm += change\n",
    "            \n",
    "            # Jaga di rentang 120 - 170\n",
    "            current_pwm = max(MIN_PWM, min(MAX_PWM, current_pwm))\n",
    "            \n",
    "            # Kirim ke Motor\n",
    "            ser.write(f\"{current_pwm}\\n\".encode())\n",
    "            \n",
    "            # Baca Data\n",
    "            if ser.in_waiting:\n",
    "                try:\n",
    "                    line = ser.readline().decode('utf-8', errors='ignore').strip()\n",
    "                    if ',' in line:\n",
    "                        parts = line.split(',')\n",
    "                        if parts[0].isdigit() and parts[1].isdigit():\n",
    "                            rec_pwm = int(parts[0])\n",
    "                            rec_rpm = int(parts[1])\n",
    "                            \n",
    "                            now = round(time.time() - start_time, 2)\n",
    "                            data_log.append([now, current_pwm, rec_rpm])\n",
    "                            \n",
    "                            # Cek status\n",
    "                            if len(data_log) % 10 == 0:\n",
    "                                status = \"OK\" if rec_rpm > 0 else \"MACET (Bantu putar pakai tangan dikit)\"\n",
    "                                print(f\"Rec: t={now}s | PWM={current_pwm} | RPM={rec_rpm} [{status}]\")\n",
    "                except:\n",
    "                    pass\n",
    "            \n",
    "            time.sleep(0.1) \n",
    "\n",
    "        # Selesai\n",
    "        ser.write(b'0\\n')\n",
    "        ser.close()\n",
    "        \n",
    "        # Simpan\n",
    "        if len(data_log) > 0:\n",
    "            df = pd.DataFrame(data_log, columns=['Waktu', 'PWM', 'RPM'])\n",
    "            df.to_csv('data_motor.csv', index=False)\n",
    "            print(f\"\\n✅ SUKSES! Data tersimpan ({len(df)} baris).\")\n",
    "        else:\n",
    "            print(\"\\n⚠️ Data Kosong.\")\n",
    "            \n",
    "    except KeyboardInterrupt:\n",
    "        ser.write(b'0\\n')\n",
    "        ser.close()\n",
    "        print(\"\\nDibatalkan user.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "9a6e153b-79c2-4105-a602-64c05ecfdfb3",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "✅ Data berhasil dimuat!\n",
      "   Waktu  PWM   RPM\n",
      "0   0.51  133  2760\n",
      "1   1.02  131  4800\n",
      "2   1.53  130  2640\n",
      "3   2.04  138  2040\n",
      "4   2.55  132  1920\n",
      "Siap melatih dengan 230 sampel data.\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "C:\\Users\\Farrel\\lstm_env\\Lib\\site-packages\\keras\\src\\layers\\rnn\\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.\n",
      "  super().__init__(**kwargs)\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "🚀 SEDANG MELATIH OTAK... (Tunggu sebentar)\n",
      "Epoch 1/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m2s\u001b[0m 8ms/step - loss: 0.3397\n",
      "Epoch 2/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0536\n",
      "Epoch 3/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 8ms/step - loss: 0.0345\n",
      "Epoch 4/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0280\n",
      "Epoch 5/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0261\n",
      "Epoch 6/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0249\n",
      "Epoch 7/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0244\n",
      "Epoch 8/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0233\n",
      "Epoch 9/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0226\n",
      "Epoch 10/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0217\n",
      "Epoch 11/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0213\n",
      "Epoch 12/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0205\n",
      "Epoch 13/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0197\n",
      "Epoch 14/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0190\n",
      "Epoch 15/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0185 \n",
      "Epoch 16/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0176\n",
      "Epoch 17/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0169\n",
      "Epoch 18/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0162\n",
      "Epoch 19/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0157\n",
      "Epoch 20/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0160\n",
      "Epoch 21/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0148\n",
      "Epoch 22/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0143\n",
      "Epoch 23/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0145\n",
      "Epoch 24/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0140\n",
      "Epoch 25/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0131\n",
      "Epoch 26/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0127\n",
      "Epoch 27/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 7ms/step - loss: 0.0123\n",
      "Epoch 28/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0126\n",
      "Epoch 29/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0125 \n",
      "Epoch 30/30\n",
      "\u001b[1m15/15\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 6ms/step - loss: 0.0116\n"
     ]
    },
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. \n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "✅ SELESAI! Model disimpan sebagai 'otak_motor.h5'\n",
      "\u001b[1m8/8\u001b[0m \u001b[32m━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[37m\u001b[0m \u001b[1m0s\u001b[0m 29ms/step\n"
     ]
    },
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1000x500 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "image/png": "iVBORw0KGgoAAAANSUhEUgAAAg8AAAEpCAYAAAAUOMEUAAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjgsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvwVt1zgAAAAlwSFlzAAAPYQAAD2EBqD+naQAAOCVJREFUeJzt3Qt4FOW5wPE32dxIAoEQIYAIIgiiEDRcREU8SkGPtajYBuopSC0eRT1aVASVS6stiMpDVQ5UrcUroq3aU0qpGg3WGkRBSpVLgWK5BkIwQBLIZbPneb/NLLthE3Y2G7LJ/H8+4+7Mzk4msxPm3fd7v29iPB6PRwAAAEIUG+qKAAAABA8AAMA2Mg8AAMAWggcAAGALwQMAALCF4AEAANhC8AAAAGwheAAAALYQPAAAAFsIHoAGWLlypQwYMECSkpIkJiZGiouL5ZZbbpHu3bv71vnmm2/Ma08++aTjjvWbb74p6enpUlJS0tS7gjpcfPHFMnXqVI4PbCF4QIuxY8cOueuuu+Tcc8+V5ORkM/Xt21fuvPNO2bBhQ8R/XlFRkfzgBz+QVq1aycKFC+WVV16RlJSUiP8cDTzqmm6//XaJVm63W2bNmiV33323pKam+pZrYPXd73436Hvy8vLM7/W73/1OmovZs2fX+xlZ0xVXXCHR6MEHHzTnb0FBQVPvCpqRuKbeASASli9fLjk5ORIXFyc333yzZGVlSWxsrGzevFnefvttWbRokQkuunXrFrED/vnnn8vRo0fl0UcflREjRviWP//881JdXS2R9J3vfEfGjx9/0nINlKLVH//4R9myZYvcdttt0pLdeOON0rNnT9+8ZlnuuOMOueGGG8xrlo4dO0o0Gj16tLRp00b+93//V37+85839e6gmSB4QLO3fft2GTt2rAkMcnNzpVOnTgGvP/744+YfRg0m6lNaWmorc3DgwAHz2LZt24Dl8fHxEmkaJPzXf/2X7feVlZWZDExtVVVVJsBJSEgIe59Odbx++9vfyqWXXipdunSRpmL3Mw1H//79zWQ5ePCgCR50WTifmZ3PMRL07+Kmm26Sl19+WX72s5+ZLAlwKjRboNmbN2+euUjoxap24KA0G/E///M/0rVrV98yrUvQVLoGHv/5n/8prVu3NhkL9de//lW+//3vy1lnnSWJiYnmfT/96U/l2LFjvvdrCnrChAnm+aBBg8w/uLpNa9v+NQ/B6M1s9Ru5Xrw1MxIJuk8XXHCBrF27Vi6//HJzsXnooYcCai4WLFgg55xzjvm9Nm7caN734YcfyrBhw8xFVgMh/Sa6adOmoKl5fc8Pf/hDadeunVx22WV17svx48dNPYh/RiZc//73v2Xy5MnSu3dv00TUvn178/no7+VvyZIlZh9XrVpl1u/QoYOceeaZ9X4m1u/l7/333ze/mx4LPUf05+pxbAhr32rvs9VMo4+n+hyVrqv7XJv+btb55//z/va3v8mUKVPkjDPOMJ+vZkMKCwuDZrb0OK9fv75Bvyecg8wDWkSThaaNhwwZYut9+u171KhR5kKhF1brm91bb71lvunpt0e9UK1Zs0aeeeYZ2b17t3lNPfzww+ai8txzz5lU79lnn20uyqHWAvz4xz+WZcuWyTvvvCPXXnvtKd+jF2P9Rlubppv9swdah3HNNdeYTIx+6/VPlWtwpdvRoEWDBy1k/OCDD8z6PXr0MBclDZD0d9WMwbp160664OpFu1evXvLLX/7SBEB10QtfRUWFXHTRRUFfr6ysDPr7HD58OGjz0Keffmp+Jw0G9AKszVB6kdVgpvY3cg0c9GI5c+ZME1Ta8fXXX5t6DM0a6Oeqx2nbtm3mInw61fc52qH1Jhroae2JHjcNHrUuSM89f9nZ2eZRf88LL7wwIr8DWjaCBzRrR44ckb1798r1119/0mva80EDBIt+89Jvrpby8nJzMZwzZ85JzRz+6+nFVoMT/fa3c+dOk5HQb2p79uwxwYP+Iz9w4MCQ9lf3Ry8G//d//2emkSNHhvS+3/zmN2aqbenSpeYCY9Git8WLF8t///d/+5ZZ33Y1+NELoV5YLZpl0CAiPz/fPCo9lnoB0QvOSy+9FPDztJbk9ddfP+X+aq2J0qAqmPfeey9gP+qjwZWm1f1dd911MnToUPn9738vP/rRjwJe099Dm69cLpfYpVkHDXr+/Oc/S0ZGhjSVYJ9jODT41WNtZVe0qerpp582QVpaWppvPW1a0iDUykYBp0KzBZp98KD8q/kt+s1UL1DWpBXltWl2oTb/wEG/ueo35EsuucR80/7yyy/D3le9KGmwopmSFStWhBw4WBd5vbDVnv7jP/4jYD39pjxx4sSg2xgzZkzABXvfvn0mTa3pbitwUPqtW4Mj3cfaQu3dod+clX7rDUazRMF+n2DdWf0/D81Y6LY1mNNmBc2O1DZp0qSwAgf/+pU//OEPES96taO+z9EODXz9m2W0eUozX9pEUZt+VsGyQUAwZB7QrGmtggo2jsCvf/1r0xti//79QQvXtBbCahP3p9kFTXlrZuDbb789ZVo9VJrh0P3Ub7V2u+3pfoZSP2B9gwymdhbAuoBo80tt5513nvzlL385qeCwrkxCXepq2tBv9cF+H/1MatOmFD122uyi2R7/bQb7POzuoz/tsfPCCy/IT37yE5k2bZpcddVVpseEZj5OVXAbSfV9jnZolsyfFczVPq+VHleKJREqggc0a5p61SLJr7766qTXrBqI2kVq/t/ual8Q9FuZfus+dOiQ6f/ep08fc/HUi5Z+Q2/It1Gtr9AiQi3w1OBBB5aKNP9v6XZei8T2a6fLrYtUsADNbru9Bg733nuvaarQz1wvctpcE+zzCLaPdV0U9fOu/d6PP/5YPvroI/nTn/5kPi+tD7jyyitN+j/cjEaoP7++36E+dW2nrv0NFtRpM19TNtWgeaHZAs2etolrW74WNjbUP/7xD/nnP/8pTz31lAketLlAvyF37tw5IiP5vfvuu6b4T5sv/OsxmoI15oWOxRCsZkEvJOF2c9SgS+nYGg2lA0Zpzxb9TDQDoMGdFrnqxS5U+o072PrB0vcaUGrGYf78+aYG4Be/+IXpkaIBRbisb/y19yHYzz/VdmpvQ5vDtAmqITQ41u1oxgkIBcEDmj0dWlcr7rUHgzZR1FZfr4C6vqn5v0ef/+pXv4rIvmog8sYbb5hvtFro15Tt6pqx0aG1tSjS/4KkWRz9lq1dWMOl1fuadv/iiy8avJ/6mdT+DLVHSF3ftoPRnjDaxOE/0qhecLW3iz/NONWmx8gqsA2X1RNHsxoW3X8tuLW7Hf9tKN2GnWNRV+8YpbU9QChotkCzp10HtQfAuHHjTPu9NcKkXnD0m6++pt8mQ0mf6zdm/Qf6/vvvN9/GtCukVvQHayMOl/Zm0DS8jhip29fajFPRbMirr7560nLtwqffxMP1xBNPmN4i2hxw6623+rpqatNAsPEEQqVNMloQql1BGzpqoXad1KG/dZ90uHHtGaLbtZpGQqFNHJpJ0nEOdMwP7Yqr3T118C3/okvdV704azZLMzM6EJgOMKbnTn3jWpzK+eefbzJP06dPNwGKFqhqEGk3+6S1GFq0qsWv+rn//e9/N7UpDW1u0GJVrY+gmyZCRfCAFkGbF7TJQVPb+q35xRdfNO3MegHQC4H+g6sBxano6JA6rLJeYLRITy+CesHRvvGhvD9UWsCpxZw6JoEGEHoRr4/VG6G24cOHNyh40EyIZkG0W6YWiervr9vU7qoNKTxUmgnSi9yuXbsCBuiyS7M+mn147bXXzDgVOgaFBg9aQxIqDTQ0y6ADJmmmSn83/Xy3bt0aEDx873vfMzUyev5ozwO9KOvx0JEX/bs2hkP3X7tezp071/Tq0GBNe8vY+fy0J4kGxNptVz837T2h54U2s4RLs18aIOv+UDCJUMV47OR0ASBEmkrXTIHePEzv/4HopHU4OmqojrYabIRWIBiCBwCNRnsq6Fga2v012FgcaHraZKUZDO0FBISK4AEAANhCbwsAAGALwQMAALCF4AEAANhC8AAAAJw3zoP2U9bbMutNkuinDABA6HTEBh13RofhD/kGcJ4wPPvss55u3bp5EhMTPYMHD/Z89tlnda77+9//3pOdne1JS0vzJCcne7Kysjwvv/xywDoTJkzQsSYCplGjRoW8P7t27Trp/UwcA84BzgHOAc4BzgEJ+RjotTRUceH029ZR2hYvXmzuWrhgwQIz0pveXKdDhw4nra/DsD788MNm2F8d63758uXmPvW6rv8IcVdffbUZstf/jod2b8usI9npaH0AACA0R44cMaPAWtfSRhnnQQOGQYMGybPPPutrMtAfqrfNnTZtWkjbuOiii8yQwdaoc3qrY70xj450Fu4vrkPH6o1vCB4AAGjca6itgkm9ZavefU3Hw/dtIDbWzOvNak5F45Tc3FyTpbj88ssDXsvLyzPZCL2xkY5IV1RUVOd29O52+sv6TwAA4PSw1WyhN4rR8er1Tn7+dH7z5s11vk+jmS5dupiLvt7gRu9S538zGG2yuPHGG83NanR89Yceesjc6U8DEusWyf70hjZ6oxoAANBCe1toO8r69eulpKTEZB60ZqJHjx5yxRVX+G6Xa+nXr5/079/f3BZZsxHB7hant7XVbdRurwEAAFEWPOjtaTUTsH///oDlOp+ZmVnn+7Rpo2fPnub5gAEDZNOmTSZ7YAUPtWlgoT9r27ZtQYMHLaa0U1AJAAAix1bNg/aWyM7ONtkDixZM6rzemS1U+h5twqjL7t27Tc0Dt4cFAKAFNFtoc8GECRNk4MCBMnjwYNNVs7S01HS/VOPHjzf1DZpZUPqo62ozhAYMK1askFdeeUUWLVpkXtemDK1fGDNmjMleaM3D1KlTTabCvysnAABopsFDTk6OFBYWysyZM6WgoMA0Q6xcudJXRLlz586AEao0sJg8ebLJJrRq1cqM9/Dqq6+a7ShtBtmwYYO89NJLprumjnA1cuRI042zKZsmCg4fl+2FJdI2OV7O75zWZPsBAEC0sT3OQzRqjHEeFq/aLnP/vFluvKiLzP/BgIhsEwAAx43z4CTpKQnm8dvSiqbeFQAAogrBQx3Sk73BwyGCBwAAAhA81KFdTebhUBmZBwAA/BE81KG9r9misq5VAABwJIKHU2QeSsqrpLzKfTo/EwAAohrBQx3aJMVJXGyMeU72AQCAEwge6hATE+PLPhSV1j0aJgAATkPwEEKPCzIPAACcQPAQwlgP9LgAAOAEgodQgocSmi0AALAQPNSjXUq8eTxURndNAAAsBA/1SE/x3piLIaoBADiB4KEe6ck1mQeGqAYAwIfgIZQhqgkeAADwIXioR/uaZguCBwAATiB4CKlgkptjAQBgIXgIIfOgBZMej6e+VQEAcAyCh3q0rSmYrKr2yJHjVafrMwEAIKoRPNQjKd4lKQku85zumgAAeBE8nEJ6KkNUAwDgj+AhxJtjHSqhaBIAgLCDh4ULF0r37t0lKSlJhgwZImvWrKlz3bffflsGDhwobdu2lZSUFBkwYIC88sorAetoMeLMmTOlU6dO0qpVKxkxYoRs3bo1Kj4hbo4FAEADg4dly5bJlClTZNasWbJu3TrJysqSUaNGyYEDB4Kun56eLg8//LDk5+fLhg0bZOLEiWb6y1/+4ltn3rx58vTTT8vixYvls88+M0GGbvP48ePS1BgoCgCABgYP8+fPl0mTJpkAoG/fvuaCn5ycLC+++GLQ9a+44gq54YYb5LzzzpNzzjlH7rnnHunfv7988sknvqzDggUL5JFHHpHRo0eb115++WXZu3evvPvuuxItzRYUTAIAEEbwUFFRIWvXrjXNCpbY2Fgzr5mFU9FAITc3V7Zs2SKXX365WbZjxw4pKCgI2GZaWpppDqlrm+Xl5XLkyJGAqdELJhmiGgAA+8HDwYMHxe12S8eOHQOW67wGAHU5fPiwpKamSkJCglx77bXyzDPPyHe+8x3zmvU+O9ucM2eOCTCsqWvXrtLoBZMEDwAAnL7eFq1bt5b169fL559/Lr/4xS9MzUReXl7Y25s+fboJSKxp165d0ug1DwxRDQCAESc2ZGRkiMvlkv379wcs1/nMzMw636dNGz179jTPtbfFpk2bTPZA6yGs9+k2tLeF/zZ13WASExPNdDq0586aAACEn3nQZofs7GxTt2Cprq4280OHDg15O/oerVtQZ599tgkg/LepNQza68LONhsLvS0AAGhA5kFpk8OECRPM2A2DBw82PSVKS0tN7ws1fvx46dKli8ksKH3UdbWnhQYMK1asMOM8LFq0yLweExMj9957rzz22GPSq1cvE0zMmDFDOnfuLNdff700NSvzcPR4lVS6qyXexbhaAABnsx085OTkSGFhoRnUSQsatWlh5cqVvoLHnTt3mmYKiwYWkydPlt27d5sBoPr06SOvvvqq2Y5l6tSpZr3bbrtNiouL5bLLLjPb1EGomlqbpHiJjRGp9ni7a3Zo0/T7BABAU4rxtIB7TWszh/a60OLJNm3aRHz72Y++L0WlFbLy3mHSJzPy2wcAoDldQ8nB2xmimu6aAAAQPISCokkAAE4g8xAChqgGAOAEggcbQ1Rr3QMAAE5H8BACMg8AAJxA8GCnYLKsMpTVAQBo0QgebPW28I6KCQCAkxE82OptQeYBAACCBxtDVOsIkwAAOB3Bg81xHlrAgJwAADQIwYON3hYV7moprXA37IgDANDMETyEoFWCS1rFu8zzQyU0XQAAnI3gwXZ3TYIHAICzETzYDB4omgQAOB3Bg82iSYaoBgA4HcFDiNKT480jmQcAgNMRPIQoPSXRPFLzAABwOoKHEKWneDMP9LYAADgdwUOIyDwAAOBF8GA388AQ1QAAhyN4CFG7mlEmKZgEADhdWMHDwoULpXv37pKUlCRDhgyRNWvW1Lnu888/L8OGDZN27dqZacSIESetf8stt0hMTEzAdPXVV0s0aZ/KIFEAAIQVPCxbtkymTJkis2bNknXr1klWVpaMGjVKDhw4EHT9vLw8GTdunHz00UeSn58vXbt2lZEjR8qePXsC1tNgYd++fb5p6dKlUZl5KC6rlCp3dVPvDgAAzSd4mD9/vkyaNEkmTpwoffv2lcWLF0tycrK8+OKLQdd/7bXXZPLkyTJgwADp06ePvPDCC1JdXS25ubkB6yUmJkpmZqZv0ixFNGmbnCAxMd7nxccqm3p3AABoHsFDRUWFrF271jQ9+DYQG2vmNasQirKyMqmsrJT09PSTMhQdOnSQ3r17yx133CFFRUV1bqO8vFyOHDkSMDU2V2yMtG1F0SQAALaCh4MHD4rb7ZaOHTsGLNf5goKCkLbx4IMPSufOnQMCEG2yePnll0024vHHH5dVq1bJNddcY35WMHPmzJG0tDTfpE0hp3OIanpcAACcLO50/rC5c+fKG2+8YbIMWmxpGTt2rO95v379pH///nLOOeeY9a666qqTtjN9+nRTd2HRzMPpCCDapyTIvwpL6XEBAHA0W5mHjIwMcblcsn///oDlOq91CvV58sknTfDw3nvvmeCgPj169DA/a9u2bUFf1/qINm3aBEyns2iSm2MBAJzMVvCQkJAg2dnZAcWOVvHj0KFD63zfvHnz5NFHH5WVK1fKwIEDT/lzdu/ebWoeOnXqJNGE23IDABBGbwttLtCxG1566SXZtGmTKW4sLS01vS/U+PHjTbOCRWsYZsyYYXpj6NgQWhuhU0lJiXldHx944AFZvXq1fPPNNyYQGT16tPTs2dN0AY3G4IGbYwEAnMx2zUNOTo4UFhbKzJkzTRCgXTA1o2AVUe7cudP0wLAsWrTI9NK46aabAraj40TMnj3bNINs2LDBBCPFxcWmmFLHgdBMhTZPRGXwwBDVAAAHi/F4PB5p5rRgUntdHD58uFHrH95et1umvPl3GdYrQ165dUij/RwAAKL5Gsq9LWygqyYAAAQPtqRzcywAAMg82EHBJAAABA9hBQ/HK6ulrKKK8wcA4EjUPNiQnOCShDjvIaPHBQDAqQgebIiJiTFDVKtvS7mzJgDAmQgewh6iurwxPg8AAKIewYNN7VNrMg9lFY3xeQAAEPUIHsLNPJQQPAAAnIngIdybY5F5AAA4FMFD2Pe3oGASAOBMBA9hD1FNwSQAwJkIHsIeoprMAwDAmQgewmy2oKsmAMCpCB7CLpgk8wAAcCaChzCDh+KyCnFXexrjMwEAIKoRPNjUNjnePGrccPgY2QcAgPMQPNgU74qVNklx5jk3xwIAOBHBQxjapyaaRwaKAgA4EcFDGNrVNF0wRDUAwIkIHsLAENUAACcLK3hYuHChdO/eXZKSkmTIkCGyZs2aOtd9/vnnZdiwYdKuXTszjRgx4qT1PR6PzJw5Uzp16iStWrUy62zdulWif4hqbo4FAHAe28HDsmXLZMqUKTJr1ixZt26dZGVlyahRo+TAgQNB18/Ly5Nx48bJRx99JPn5+dK1a1cZOXKk7Nmzx7fOvHnz5Omnn5bFixfLZ599JikpKWabx48fl+geoprgAQDgPDEe/dpvg2YaBg0aJM8++6yZr66uNgHB3XffLdOmTTvl+91ut8lA6PvHjx9vsg6dO3eW++67T+6//36zzuHDh6Vjx46yZMkSGTt27Cm3eeTIEUlLSzPva9OmjTS25z7eLr9csVluvLCLzM8Z0Og/DwCAxhLONdRW5qGiokLWrl1rmhV8G4iNNfOaVQhFWVmZVFZWSnp6upnfsWOHFBQUBGxTfwkNUuraZnl5ufll/afTqV3N/S2KyDwAABzIVvBw8OBBkznQrIA/ndcAIBQPPvigyTRYwYL1PjvbnDNnjgkwrEkzH6cTBZMAACc7rb0t5s6dK2+88Ya88847ptgyXNOnTzfpFWvatWuXnE4UTAIAnMw7VGKIMjIyxOVyyf79+wOW63xmZma9733yySdN8PDBBx9I//79fcut9+k2tLeF/zYHDAheT5CYmGimpkLwAABwMluZh4SEBMnOzpbc3FzfMi2Y1PmhQ4fW+T7tTfHoo4/KypUrZeDAgQGvnX322SaA8N+m1jBor4v6ttmUrOChrMItxyvdTb07AABEb+ZBaTfNCRMmmCBg8ODBsmDBAiktLZWJEyea17UHRZcuXUxdgnr88cfNGA6vv/66GRvCqmNITU01U0xMjNx7773y2GOPSa9evUwwMWPGDFMXcf3110s0Sk2Mk3hXjFS6Paa7Zue2rZp6lwAAiN7gIScnRwoLC01AoIGANi1oRsEqeNy5c6fpgWFZtGiR6aVx0003BWxHx4mYPXu2eT516lQTgNx2221SXFwsl112mdlmQ+oiGpMGPNrj4sDRcoIHAIDj2B7nIRqd7nEe1NULPpbNBUfllVsHy7BeZ5yWnwkAQLMb5wEnUDQJAHAqgocwMUQ1AMCpCB7C1L6mx8W3jDIJAHAYgocwMUQ1AMCpCB7C1D61JvNQxp01AQDOQvDQ0MxDCcEDAMBZCB7CxM2xAABORfDQ4K6alZH8PAAAiHoEDxHIPFRXN/txtgAACBnBQ5jaJsebR3e1R44erwp3MwAANDsED2FKjHNJ60TvrUGKSssj+ZkAABDVCB4iMMok3TUBAE5C8NAAFE0CAJyI4CEiwQPNFgAA5yB4iMBAUXTXBAA4CcFDAzBENQDAiQgeGoAhqgEATkTwEInbcnNzLACAgxA8RKCrZlEpN8cCADgHwUMDpKd4R5n8luABAOAgBA8NkJ6SaB4JHgAAThJW8LBw4ULp3r27JCUlyZAhQ2TNmjV1rvv111/LmDFjzPoxMTGyYMGCk9aZPXu2ec1/6tOnj0S79JqumkfLq6S8yt3UuwMAQHQGD8uWLZMpU6bIrFmzZN26dZKVlSWjRo2SAwcOBF2/rKxMevToIXPnzpXMzMw6t3v++efLvn37fNMnn3wi0a51Upy4YmPM8+Iybs0NAHAG28HD/PnzZdKkSTJx4kTp27evLF68WJKTk+XFF18Muv6gQYPkiSeekLFjx0piojfNH0xcXJwJLqwpIyNDol1sbIzfQFEUTQIAnMFW8FBRUSFr166VESNGnNhAbKyZz8/Pb9CObN26VTp37myyFDfffLPs3LmzznXLy8vlyJEjAVNTF00SPAAAnMJW8HDw4EFxu93SsWPHgOU6X1BQEPZOaN3EkiVLZOXKlbJo0SLZsWOHDBs2TI4ePRp0/Tlz5khaWppv6tq1qzT9/S3IPAAAnCEqeltcc8018v3vf1/69+9v6idWrFghxcXF8uabbwZdf/r06XL48GHftGvXLmkqBA8AAKeJs7Oy1iG4XC7Zv39/wHKdr68Y0q62bdvKueeeK9u2bQv6utZO1Fc/cTpR8wAAcBpbmYeEhATJzs6W3Nxc37Lq6mozP3To0IjtVElJiWzfvl06deok0Y4hqgEATmMr86C0m+aECRNk4MCBMnjwYDNuQ2lpqel9ocaPHy9dunQxdQlWkeXGjRt9z/fs2SPr16+X1NRU6dmzp1l+//33y3XXXSfdunWTvXv3mm6gmuEYN26cRDuGqAYAOI3t4CEnJ0cKCwtl5syZpkhywIABptDRKqLUXhLaA8OiwcCFF17om3/yySfNNHz4cMnLyzPLdu/ebQKFoqIiOeOMM+Syyy6T1atXm+fRzqp5YJRJAIBTxHg8Ho80c9pVU3tdaPFkmzZtTuvP/uvWQvnRb9ZIn8zWsvLey0/rzwYAoCmuoVHR26I5o2ASAOA0BA8N1D61ptmirEJaQBIHAIBTIniIUOah0u0xN8gCAKClI3hooKR4lyQnuMxziiYBAE5A8BABjDIJAHASgocIIHgAADgJwUMEEDwAAJyE4CEC0muKJrmzJgDACQgeIjhE9aEybssNAGj5CB4igCGqAQBOQvAQAdQ8AACchOAhAhiiGgDgJAQPER2iujISmwMAIKoRPEQw81BUUh6JzQEAENUIHiKgfU1viyPHq6TSXR2JTQIAELUIHiKgTat4iY0R3901AQBoyQgeIsAVGyNta5ouvi2l7gEA0LIRPEQI3TUBAE5B8BAhDFENAHAKgocIaZcSbx4ZohoA0NIRPERIekqiefy2lIJJAEDLFlbwsHDhQunevbskJSXJkCFDZM2aNXWu+/XXX8uYMWPM+jExMbJgwYIGbzMapVuZB4IHAEALZzt4WLZsmUyZMkVmzZol69atk6ysLBk1apQcOHAg6PplZWXSo0cPmTt3rmRmZkZkm9GceSB4AAC0dLaDh/nz58ukSZNk4sSJ0rdvX1m8eLEkJyfLiy++GHT9QYMGyRNPPCFjx46VxMTEiGwzGpF5AAA4ha3goaKiQtauXSsjRow4sYHYWDOfn58f1g6Es83y8nI5cuRIwNTUuDkWAMApbAUPBw8eFLfbLR07dgxYrvMFBQVh7UA425wzZ46kpaX5pq5du0pTa28VTDLCJACghWuWvS2mT58uhw8f9k27du2Kmq6aRaUV4vF4mnp3AABoNHF2Vs7IyBCXyyX79+8PWK7zdRVDNsY2tXairvqJph5hsqKqWsoq3JKSaOvQAgDQMjMPCQkJkp2dLbm5ub5l1dXVZn7o0KFh7UBjbLMpJCfESVK893DS4wIA0JLZ/nqsXSonTJggAwcOlMGDB5txG0pLS01PCTV+/Hjp0qWLqUuwCiI3btzoe75nzx5Zv369pKamSs+ePUPaZnMaonrv4eMmeOiantzUuwMAQHQEDzk5OVJYWCgzZ840BY0DBgyQlStX+goed+7caXpLWPbu3SsXXnihb/7JJ5800/DhwyUvLy+kbTYX6ak1wQNFkwCAFizG0wKq+7Srpva60OLJNm3aNNl+/Og3n8lftx6Up76fJWOyz2yy/QAAoDGvoc2yt0W0soom6a4JAGjJCB4aIXigYBIA0JIRPES4YFIRPAAAWjKChwgXTCqCBwBAS0bwEEFkHgAATkDwEEHtrJoHumoCAFowgocIam/1tiitiORmAQCIKgQPjZB5KD5WKe7qZj98BgAAQRE8RFDbVt47a+qwW8U0XQAAWiiChwiKc8VK22RvAMFAUQCAlorgoZF6XBSVUPcAAGiZCB4ijCGqAQAtHcFDIxVNFtHjAgDQQhE8NFKzBd01AQAtFcFDow1RXRnpTQMAEBUIHhptiOrySG8aAICoQPDQaENUk3kAALRMBA8RxhDVAICWjuChsTIP9LYAALRQBA+NlHkgeAAAtFQED42UeThW6ZZjFe5Ibx4AgOYZPCxcuFC6d+8uSUlJMmTIEFmzZk2967/11lvSp08fs36/fv1kxYoVAa/fcsstEhMTEzBdffXV0hylJLgkweU9rIe4ORYAoAWyHTwsW7ZMpkyZIrNmzZJ169ZJVlaWjBo1Sg4cOBB0/U8//VTGjRsnt956q3z55Zdy/fXXm+mrr74KWE+DhX379vmmpUuXSnOkgY81RPUHG/c39e4AABBxMR6P3kA6dJppGDRokDz77LNmvrq6Wrp27Sp33323TJs27aT1c3JypLS0VJYvX+5bdvHFF8uAAQNk8eLFvsxDcXGxvPvuu2H9EkeOHJG0tDQ5fPiwtGnTRpraL/60UZ7/6w7z/Pbh58jUUb0lNjamqXcLAICIXENtZR4qKipk7dq1MmLEiBMbiI018/n5+UHfo8v911eaqai9fl5ennTo0EF69+4td9xxhxQVFdW5H+Xl5eaX9Z+iyfRrzpO7r+xpni9etV1uf3WtlFVUNfVuAQAQEbaCh4MHD4rb7ZaOHTsGLNf5goKCoO/R5adaX5ssXn75ZcnNzZXHH39cVq1aJddcc435WcHMmTPHREnWpJmPaKJZhvtG9pYFOQNM/cN7G/fLTYvyZd/hY029awAAtIzeFmPHjpXvfe97pphS6yG0iePzzz832Yhgpk+fbtIr1rRr1y6JRtdf2EWW3naxZKQmyMZ9R+R7z/5N/r6ruKl3CwCA0xc8ZGRkiMvlkv37AwsBdT4zMzPoe3S5nfVVjx49zM/atm1b0NcTExNNu4z/FK2yu7WTd++8VHp3bC2FR8vlB7/Ol+Ub9jb1bgEAcHqCh4SEBMnOzjbNCxYtmNT5oUOHBn2PLvdfX73//vt1rq92795tah46deokLcGZ7ZLl95MvkSv7dJDyqmq56/Uv5encrWKzVhUAgObZbKHdNJ9//nl56aWXZNOmTaa4UXtTTJw40bw+fvx406xgueeee2TlypXy1FNPyebNm2X27NnyxRdfyF133WVeLykpkQceeEBWr14t33zzjQk0Ro8eLT179jSFlS1FamKcPD9+oNx62dlmfv77/5R7l62X45UMJAUAaF7i7L5Bu14WFhbKzJkzTdGjdrnU4MAqity5c6fpgWG55JJL5PXXX5dHHnlEHnroIenVq5fpknnBBReY17UZZMOGDSYY0e6anTt3lpEjR8qjjz5qmidaEldsjMz4bl/p2SFVZrz7lfxh/V7ZeahMnvvRQDmjdcv6XQEALZftcR6iUbSN8xCKT7cdlDteWyeHj1VKl7at5IUJA+W8Ts1j3wEALUejj/OAyLmkZ4a8M/kS6ZGRInuKj8lNiz6V3E2MSAkAiH4ED02oxxmp8s7kS+WSc9pLaYVbfvLyF/LCX/9FISUAIKoRPDSxtOR4eenHg2Xc4LNEG5Ae+9Mmyfn1avn1qu2ypeAogQQAIOpQ8xAltPTkt3/7Rh7700ap9qtC6ZyWJMN7nyHDzz1DLu2ZIa2T4ptyNwEALUw4NQ8ED1FmZ1GZ5G7eL3lbCmX1v4rMuBCWuNgYM+jUFb07mGDivE6tzV08AQAIF8FDM+ptEQodAyL/X0WyakuhrPpnoew4WBrwesc2iSaI0GBCsxJprchKAADsIXhoYcFDbf8uKjVBhGYlPt1+UI5XVgeMIdGvS5p0bpsk7VMSJT0lwdxTo31qorRP8T7qfJukeG4PDgDwIXho4cFD7azE598cMoFE3pYDsr0wMCtRF236aKfBhAkuEqW9BhgpiSZrkZLoMiNhpiTGSWpSnPd5gvdR5/X1xDhXY/9qAIDTiODBQcFDbbsOlcn6XcVSVFIuRaUV3kmfl3ifHywpl6PHqxr8c+JdMSa4SKkJKjSgSIiLlXhXrCTWPFpTQlxM4LwrxreuWT8+1mzHuz2XJGuQkuiSZF2WECfJiS6zHgAguoIH28NTIzp1TU82U33Kq9zybWmlCSSs4OJQaYUUlpTLkWNVUlrunUr0saJKSo7rc7dZdqzmHhyVbo8Ul1Wa6XTQYMMEFjXBigYUKQlxkhSvwYkGLJoN8QYuAfP6uglQrNdPLE+Kd0lSnEtaJXiXt0pw1SyLlTiCFQA4JYIHB9ELZWaaTkm23+uu9vgCCl+AUe42j5Xuat9U4fZIZVXN8yq/Zf7rVHnnNSApq/Bup/Zjhdtbz6Hb0Onb0xSsaGbFBBLxLmllHmPNowYhGozo63HWY6wGGzESbz26Yk2zUO3XvUFMrAlY9NEXzMSfCGr059RephO9aQBEI4IHhEQLMrXYUqfTQQOGYxVuE7BoQKEZkDKTEfFmQjSLouuUW1Ol+8RzM9XMV1YHrKu1IuU12z5e5TaP/t1hNbNS6a6KSBNPJGhQYQUy3mDGmykJfO4NcJJqlluvWdkYM5lmpMDsTO3l3te8ywhaANSH4AFRybqY6Qicja262uMLLKyAQnuyaGZEg5JjNZM3c+KRKp2qrefVUlXtzaTo8srqmkffulb2xRvI6Pa9AY03qNGf4x/o6Ov+t6rT13UqltOTebFYmZP4WnUqOu+tZ4n1ZVpOvO7N2iSbICbONAtpc5MJaEzTkyvguXfdON9zK3CJjWXsEiDaETzA8fRiZb7BJ7iiYqRRDTr8AwsNao5VeIMZM5ng5sRzffSucyLQ0Xn/zIzV/KNBjHe5O2CZ/kx/3gyMbuv0HwMNQjQ74suE1KppsZqBrJqWwPlgdTA176v1/ETmJTAg8hb3eoMhzbg1JAujganb4zHNfhpkVns8JoCiEBjNHcEDEEX0QqUXM72QtbZfmtKgi5zJjtQEFCazUuVd5v/cv3bFG2DUPK/ySLnJrmjdil9gU+GWMvPcW3RrXqt53XquzVL+Q7JbTUdSLk1O44YEK5gwwYU30NDl1dVijo2W57g146TBQU2Q4K4JGvyzSP708zUFwAknukebLtI1BcHe594CYavLtK4bEODoeeJySbwV/AQEQCeW1c7kaICqx1sDmeqaffT4zetr1jpKAyhvLY+3vofMEBTBAwBzQUiK9TYfNFW2xdu04y2W1SacE481TT0By+tez5o32ZVa61hNRFZWxmpS8hb5evchcN+0l5J33UgGM/rzD1VVyKHQhmdpEL3wKyswaCgNnKwiYQ0sfIXCNcXCulyfWz2ZNNNiNVlpXU5yTdPViSauE+tZj2Y71rZrghaXeTzxM6x90PWo0Tn9CB4ARE22Rb9lNyUNZKwaFv8mHV8Poprleg3Wi1ZsjPcbuSvGeyHVOpHYWAl4NK/VXOSUNin591bSAmCrKPhET6aaqeLEMs3SVGqGqGZfAno0+e2rZj386e8T2WMk3uPg7b0dFfTYWoGMlSnxPdbU75y0vObz0kedPOI9dppR0kBLM0eaSdLDZ5bXZGa8z8X33MouaVBltXDF6H8673eOm+e+173LtBnNCpr8C6N1Xpvg/Odrv96rQ6p0aHMa05O1EDwAQA39B91K+ScnNM5h0YtA28baeE23av+Mij7qpUtjF/399FGDHp1iYq3n3scYv9d0mV4YrWYYqxjYKhL2LxCuOunRW7djNU0d92/O0poc01x1oj7Hv6lL502xsV/xsfXzdFmwWMi8XlP47BSPj+knOYPOarKfT/AAAC2I95t0ZJqgTM1HTcaklTR9QbGqDghkTg5gfAWqNVkYb13KiVoU72PNun7LrcBJj19swPMY06yny1y+5zGi48l5g7EYk7FS+v8TdS419STWXE0tif96GtiFWgB9Yr7aBF96W4GmRPAAAGg29OKdGKs9app6T5yNGwcAAIDGDx4WLlwo3bt3l6SkJBkyZIisWbOm3vXfeust6dOnj1m/X79+smLFioDXNZUzc+ZM6dSpk7Rq1UpGjBghW7duDWfXAABAtAUPy5YtkylTpsisWbNk3bp1kpWVJaNGjZIDBw4EXf/TTz+VcePGya233ipffvmlXH/99Wb66quvfOvMmzdPnn76aVm8eLF89tlnkpKSYrZ5/Pjxhv12AAAg4mI8VgVHiDTTMGjQIHn22WfNfHV1tXTt2lXuvvtumTZt2knr5+TkSGlpqSxfvty37OKLL5YBAwaYYEF/fOfOneW+++6T+++/37yutwXt2LGjLFmyRMaOHdsotxMFAAAS1jXUVuahoqJC1q5da5oVfBuIjTXz+fn5Qd+jy/3XV5pVsNbfsWOHFBQUBKyjv4QGKXVts7y83Pyy/hMAADg9bAUPBw8eFLfbbbIC/nReA4BgdHl961uPdrY5Z84cE2BYk2Y+AADA6dEse1tMnz7dpFesadeuXU29SwAAOIatnrIZGRnicrlk//79Act1PjMzM+h7dHl961uPukx7W/ivo3URwSQmJprJYpVt0HwBAIA91rXTTgmkreAhISFBsrOzJTc31/SYsAomdf6uu+4K+p6hQ4ea1++9917fsvfff98sV2effbYJIHQdK1jQX0R7Xdxxxx0h7dfRo0fNI80XAACER6+lWgoQCttjdGk3zQkTJsjAgQNl8ODBsmDBAtObYuLEieb18ePHS5cuXUxdgrrnnntk+PDh8tRTT8m1114rb7zxhnzxxRfy3HPPmdd1eE8NLB577DHp1auXCSZmzJhhemBYAcqp6LradNG6deuI3l1NgxgNSHTb9OLg+HAONQ7+zjhGnEdN+7emGQcNHPRaGirbwYN2vSwsLDSDOmlBo2YLVq5c6St43Llzp+mBYbnkkkvk9ddfl0ceeUQeeughEyC8++67csEFF/jWmTp1qglAbrvtNikuLpbLLrvMbFMHlQqF/rwzzzxTGoseZIIHjg/nUOPi74xjxHnUdH9roWYcwh7nwUkYP4LjwznE31k04N8ijlG0nUfNsrcFAABoOgQP9dAeHToMt3/PDnB87OAc4hhFAucRxyjaziOaLQAAgC1kHgAAgC0EDwAAwBaCBwAAYAvBAwAAsIXgoQ4LFy6U7t27m4Gq9Pbga9assXdkW7DZs2ebkTz9pz59+oiTffzxx3LdddeZEdr0eOhAaP50OBUdWE3v39KqVStzC/qtW7eKk5zqGN1yyy0nnVdXX321OIWOyjto0CAzUm6HDh3MCLtbtmwJWOf48eNy5513Svv27SU1NVXGjBlz0r2DnH6MrrjiipPOo9tvv12cYtGiRdK/f3/fQFB6K4g///nPET+HCB6CWLZsmRmGW7u0rFu3TrKysmTUqFFy4MCBhn2qLcj5558v+/bt802ffPKJOJmOkKrniQadwcybN0+efvppWbx4sblvS0pKijmn9A/ZKU51jJQGC/7n1dKlS8UpVq1aZf5RX716tbn/T2VlpYwcOdIcN8tPf/pT+eMf/yhvvfWWWX/v3r1y4403ilOEcozUpEmTAs4j/ftzijPPPFPmzp0ra9euNbeCuPLKK2X06NHy9ddfR/Yc0hEmEWjw4MGeO++80zfvdrs9nTt39syZM4dD5fF4Zs2a5cnKyuJY1EH/rN555x3ffHV1tSczM9PzxBNP+JYVFxd7EhMTPUuXLnXkcax9jNSECRM8o0ePbrJ9ijYHDhwwx2nVqlW+cyY+Pt7z1ltv+dbZtGmTWSc/P9/jRLWPkRo+fLjnnnvuadL9ijbt2rXzvPDCCxE9h8g81FJRUWEiNk0r+987Q+fz8/MbGBO2HJpy1/Rzjx495Oabbzb3NEFwO3bsMPeB8T+ndIhYbQ7jnAqUl5dn0tG9e/c2d9UtKipy7GmlQwir9PR086j/Luk3bf/zSJsLzzrrLMeeR7WPkeW1116TjIwMcw+l6dOnS1lZmTiR2+02N6PUzIw2X0TyHLJ9Y6yW7uDBg+aAWzf6suj85s2bm2y/oole9JYsWWL+gdeU4M9+9jMZNmyYfPXVV6YtEoE0cFDBzinrNXibLDR9qnfW3b59u7mR3jXXXGP+UXO5XI46RNXV1eZuw5deeqnvJoJ6riQkJEjbtm0D1nXqeRTsGKkf/vCH0q1bN/PlZsOGDfLggw+auoi3335bnOIf//iHCRa0WVTrGt555x3p27evrF+/PmLnEMEDbNN/0C1amKPBhP6xvvnmm3LrrbdyRBGWsWPH+p7369fPnFvnnHOOyUZcddVVjjqq2q6vwbjTa4nCOUZ6d2b/80iLlPX80YBUzycn6N27twkUNDPzu9/9TiZMmGDqGyKJZotaNNWl33JqV5/qfGZmZkQPfkuhUey5554r27Zta+pdiUrWecM5ZY82ienfo9POq7vuukuWL18uH330kSl+8z+PtFm1uLhYnP5vU13HKBj9cqOcdB4lJCRIz549JTs72/RQ0ULlX/3qVxE9hwgeghx0PeC5ubkB6TGd1zQQTlZSUmKieo3wcTJNw+sfpv85pbfG1V4XnFN12717t6l5cMp5pXWkelHUFPOHH35ozht/+u9SfHx8wHmk6XitN3LKeXSqYxSMfgNXTjmPgtFrWHl5eWTPoUYo7Gz23njjDVMJv2TJEs/GjRs9t912m6dt27aegoKCpt61qHDfffd58vLyPDt27PD87W9/84wYMcKTkZFhKp+d6ujRo54vv/zSTPpnNX/+fPP83//+t3l97ty55hz6wx/+4NmwYYPpVXD22Wd7jh075nGK+o6Rvnb//febim89rz744APPRRdd5OnVq5fn+PHjHie44447PGlpaeZva9++fb6prKzMt87tt9/uOeusszwffvih54svvvAMHTrUTE5xqmO0bds2z89//nNzbPQ80r+3Hj16eC6//HKPU0ybNs30PtHfX/+t0fmYmBjPe++9F9FziOChDs8884w5wAkJCabr5urVqxv2ibYgOTk5nk6dOplj06VLFzOvf7RO9tFHH5kLYu1Jux9a3TVnzJjh6dixowlMr7rqKs+WLVs8TlLfMdJ//EeOHOk544wzTFeybt26eSZNmuSogD3YsdHpt7/9rW8dDTYnT55sut4lJyd7brjhBnPxdIpTHaOdO3eaQCE9Pd38nfXs2dPzwAMPeA4fPuxxih//+Mfm70f/fda/J/23xgocInkOcUtuAABgCzUPAADAFoIHAABgC8EDAACwheABAADYQvAAAABsIXgAAAC2EDwAAABbCB4AAIAtBA8AAMAWggcAAGALwQMAALCF4AEAAIgd/w9O7zO61OO7EgAAAABJRU5ErkJggg==",
      "text/plain": [
       "<Figure size 600x300 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "from sklearn.preprocessing import MinMaxScaler\n",
    "from tensorflow.keras.models import Sequential\n",
    "from tensorflow.keras.layers import LSTM, Dense\n",
    "\n",
    "# --- 1. LOAD DATA ---\n",
    "try:\n",
    "    df = pd.read_csv('data_motor.csv')\n",
    "    print(\"✅ Data berhasil dimuat!\")\n",
    "    print(df.head()) # Lihat 5 data awal\n",
    "except:\n",
    "    print(\"❌ File 'data_motor.csv' tidak ditemukan! Pastikan file ada di folder yang sama.\")\n",
    "    exit()\n",
    "\n",
    "# Kita hanya butuh kolom RPM untuk dipelajari\n",
    "data_rpm = df['RPM'].values.reshape(-1, 1)\n",
    "\n",
    "# --- 2. NORMALISASI DATA ---\n",
    "# LSTM bekerja paling bagus dengan angka 0 sampai 1\n",
    "# Kita simpan scaler ini, karena nanti dipakai untuk prediksi real-time\n",
    "scaler = MinMaxScaler(feature_range=(0, 1))\n",
    "data_scaled = scaler.fit_transform(data_rpm)\n",
    "\n",
    "# --- 3. PERSIAPAN DATA LATIH (SEQUENCE) ---\n",
    "# Konsep: \"Lihat 10 data ke belakang (X), tebak 1 data ke depan (Y)\"\n",
    "LOOK_BACK = 10 \n",
    "X, Y = [], []\n",
    "\n",
    "if len(data_scaled) <= LOOK_BACK:\n",
    "    print(\"❌ Data terlalu sedikit! Rekam ulang minimal 1 menit.\")\n",
    "    exit()\n",
    "\n",
    "for i in range(len(data_scaled) - LOOK_BACK):\n",
    "    X.append(data_scaled[i : i + LOOK_BACK, 0])\n",
    "    Y.append(data_scaled[i + LOOK_BACK, 0])\n",
    "\n",
    "X, Y = np.array(X), np.array(Y)\n",
    "\n",
    "# Ubah bentuk array agar diterima LSTM [Samples, TimeSteps, Features]\n",
    "X = X.reshape(X.shape[0], X.shape[1], 1)\n",
    "\n",
    "print(f\"Siap melatih dengan {len(X)} sampel data.\")\n",
    "\n",
    "# --- 4. MEMBANGUN OTAK LSTM ---\n",
    "model = Sequential()\n",
    "# Layer 1: LSTM\n",
    "model.add(LSTM(units=50, return_sequences=False, input_shape=(LOOK_BACK, 1)))\n",
    "# Layer 2: Output (Menebak 1 angka RPM)\n",
    "model.add(Dense(units=1))\n",
    "\n",
    "model.compile(optimizer='adam', loss='mean_squared_error')\n",
    "\n",
    "# --- 5. MULAI TRAINING ---\n",
    "print(\"\\n🚀 SEDANG MELATIH OTAK... (Tunggu sebentar)\")\n",
    "# epochs=30 artinya dia belajar mengulang materi 30 kali\n",
    "history = model.fit(X, Y, epochs=30, batch_size=16, verbose=1)\n",
    "\n",
    "# --- 6. SIMPAN MODEL ---\n",
    "model.save('otak_motor.h5')\n",
    "print(\"\\n✅ SELESAI! Model disimpan sebagai 'otak_motor.h5'\")\n",
    "\n",
    "# --- 7. LIHAT HASIL BELAJAR ---\n",
    "# Prediksi ulang data yang sudah dipelajari untuk melihat kecocokan\n",
    "prediksi = model.predict(X)\n",
    "prediksi_asli = scaler.inverse_transform(prediksi) # Balikin ke angka RPM asli\n",
    "y_asli = scaler.inverse_transform(Y.reshape(-1, 1))\n",
    "\n",
    "# Plot Grafik\n",
    "plt.figure(figsize=(10, 5))\n",
    "plt.plot(y_asli, color='blue', label='RPM Asli')\n",
    "plt.plot(prediksi_asli, color='red', linestyle='--', label='Prediksi LSTM')\n",
    "plt.title('Hasil Training: Apakah Garis Merah Mengikuti Biru?')\n",
    "plt.legend()\n",
    "plt.show()\n",
    "\n",
    "# Plot Error\n",
    "plt.figure(figsize=(6, 3))\n",
    "plt.plot(history.history['loss'])\n",
    "plt.title('Grafik Error (Harus Turun)')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "e2bc7166-eeb2-4d32-8448-d30c3a670bb9",
   "metadata": {},
   "outputs": [
    {
     "name": "stderr",
     "output_type": "stream",
     "text": [
      "WARNING:absl:Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "⚙️ Memuat Sistem...\n",
      "✅ Otak AI Siap.\n",
      "✅ BERHASIL TERHUBUNG KE COM9\n"
     ]
    },
    {
     "data": {
      "application/vnd.jupyter.widget-view+json": {
       "model_id": "e433567c5a454c8cbcb5ed91d1d17cbe",
       "version_major": 2,
       "version_minor": 0
      },
      "text/plain": [
       "HBox(children=(IntSlider(value=0, description='Gas:', max=170), Button(button_style='danger', description='STO…"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "application/vnd.jupyter.widget-view+json": {
       "model_id": "2e5af4c4eab54c838678af79ab9520ba",
       "version_major": 2,
       "version_minor": 0
      },
      "image/png": "...",
      "text/html": [
       "\n",
       "            <div style=\"display: inline-block;\">\n",
       "                <div class=\"jupyter-widgets widget-label\" style=\"text-align: center;\">\n",
       "                    Figure\n",
       "                </div>\n",
       "                <img src='...' width=900.0/>\n",
       "            </div>\n",
       "        "
      ],
      "text/plain": [
       "Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "%matplotlib widget\n",
    "\n",
    "import serial\n",
    "import time\n",
    "import threading\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "from matplotlib.animation import FuncAnimation\n",
    "from collections import deque\n",
    "import ipywidgets as widgets\n",
    "from IPython.display import display\n",
    "from tensorflow.keras.models import load_model\n",
    "from sklearn.preprocessing import MinMaxScaler\n",
    "\n",
    "# --- KONFIGURASI ---\n",
    "PORT = 'COM9'   # <--- PASTIKAN INI BENAR!\n",
    "BAUDRATE = 115200\n",
    "LOOK_BACK = 10 \n",
    "\n",
    "# --- 1. LOAD AI ---\n",
    "print(\"⚙️ Memuat Sistem...\")\n",
    "try:\n",
    "    model = load_model('otak_motor.h5')\n",
    "    \n",
    "    # Setup Scaler Dummy/Asli\n",
    "    try:\n",
    "        df_ref = pd.read_csv('data_motor.csv')\n",
    "        scaler = MinMaxScaler(feature_range=(0, 1))\n",
    "        scaler.fit(df_ref['RPM'].values.reshape(-1, 1))\n",
    "    except:\n",
    "        scaler = MinMaxScaler(feature_range=(0, 1))\n",
    "        scaler.fit(np.array([[0], [3500]]))\n",
    "        \n",
    "    print(\"✅ Otak AI Siap.\")\n",
    "except Exception as e:\n",
    "    print(f\"❌ Error AI: {e}\")\n",
    "\n",
    "# --- 2. KONEKSI SERIAL (DEBUG) ---\n",
    "ser = None\n",
    "try:\n",
    "    # Paksa tutup koneksi lama jika ada\n",
    "    try:\n",
    "        temp_ser = serial.Serial(PORT, BAUDRATE)\n",
    "        temp_ser.close()\n",
    "    except:\n",
    "        pass\n",
    "\n",
    "    # Buka koneksi baru\n",
    "    ser = serial.Serial(PORT, BAUDRATE, timeout=1)\n",
    "    time.sleep(2)\n",
    "    ser.reset_input_buffer()\n",
    "    print(f\"✅ BERHASIL TERHUBUNG KE {PORT}\")\n",
    "\n",
    "except Exception as e:\n",
    "    print(f\"\\n🔥🔥🔥 GAGAL KONEKSI SERIAL! 🔥🔥🔥\")\n",
    "    print(f\"Penyebab: {e}\")\n",
    "    print(\"SOLUSI: 1. Tutup Serial Monitor Arduino. 2. Restart Kernel. 3. Cek kabel USB.\\n\")\n",
    "\n",
    "# --- 3. LOGIC UTAMA ---\n",
    "if ser: # Hanya jalan kalau serial connect\n",
    "    \n",
    "    # Data Storage\n",
    "    max_len = 100\n",
    "    data_asli = deque([0]*max_len, maxlen=max_len)\n",
    "    data_prediksi = deque([0]*max_len, maxlen=max_len)\n",
    "    buffer_ai = deque([0]*LOOK_BACK, maxlen=LOOK_BACK)\n",
    "\n",
    "    is_running = True\n",
    "\n",
    "    # Thread Baca Data\n",
    "    def worker_thread():\n",
    "        while is_running and ser and ser.is_open:\n",
    "            try:\n",
    "                if ser.in_waiting:\n",
    "                    line = ser.readline().decode('utf-8', errors='ignore').strip()\n",
    "                    if ',' in line:\n",
    "                        parts = line.split(',')\n",
    "                        if len(parts) > 1 and parts[1].isdigit():\n",
    "                            rpm_now = int(parts[1])\n",
    "                            \n",
    "                            # Simpan Data Asli\n",
    "                            data_asli.append(rpm_now)\n",
    "                            \n",
    "                            # AI Prediksi\n",
    "                            rpm_scaled = scaler.transform([[rpm_now]])[0][0]\n",
    "                            buffer_ai.append(rpm_scaled)\n",
    "                            \n",
    "                            if len(buffer_ai) == LOOK_BACK:\n",
    "                                input_ai = np.array(buffer_ai).reshape(1, LOOK_BACK, 1)\n",
    "                                tebakan_scaled = model.predict(input_ai, verbose=0)\n",
    "                                tebakan_rpm = scaler.inverse_transform(tebakan_scaled)[0][0]\n",
    "                                data_prediksi.append(tebakan_rpm)\n",
    "                            else:\n",
    "                                data_prediksi.append(rpm_now)\n",
    "            except:\n",
    "                pass\n",
    "            time.sleep(0.01)\n",
    "\n",
    "    t = threading.Thread(target=worker_thread)\n",
    "    t.daemon = True\n",
    "    t.start()\n",
    "\n",
    "   # --- BAGIAN GRAFIK (YANG DIPERBAIKI) ---\n",
    "    fig, ax = plt.subplots(figsize=(9, 4))\n",
    "    line_asli, = ax.plot([], [], 'b-', label='Real', linewidth=1.5)\n",
    "    line_pred, = ax.plot([], [], 'r--', label='AI Prediction', linewidth=1.5)\n",
    "    \n",
    "    # KITA NAIKKAN BATAS ATASNYA JADI 6000 (Atau 10000 biar aman)\n",
    "    ax.set_ylim(0, 6000) \n",
    "    ax.set_xlim(0, max_len)\n",
    "    ax.legend(loc='upper left')\n",
    "    ax.grid(True)\n",
    "    ax.set_title(\"Real-time RPM Monitoring\")\n",
    "\n",
    "    def update_graph(frame):\n",
    "        # Update data garis\n",
    "        line_asli.set_data(range(len(data_asli)), data_asli)\n",
    "        line_pred.set_data(range(len(data_prediksi)), data_prediksi)\n",
    "        \n",
    "        # --- LOGIKA AUTO ZOOM-OUT (TAMBAHAN) ---\n",
    "        # Cari nilai tertinggi saat ini (antara Asli atau Prediksi)\n",
    "        if len(data_asli) > 0:\n",
    "            max_asli = max(data_asli)\n",
    "            max_pred = max(data_prediksi)\n",
    "            tertinggi = max(max_asli, max_pred)\n",
    "            \n",
    "            # Ambil batas atap grafik sekarang\n",
    "            batas_sekarang = ax.get_ylim()[1]\n",
    "            \n",
    "            # Kalau grafik mau nabrak atap, tinggikan atapnya!\n",
    "            if tertinggi >= (batas_sekarang - 500):\n",
    "                ax.set_ylim(0, tertinggi + 2000) # Tambah ruang kosong di atas\n",
    "                ax.figure.canvas.draw()          # Gambar ulang kotak grafiknya\n",
    "\n",
    "        return line_asli, line_pred\n",
    "    # Slider\n",
    "    slider = widgets.IntSlider(value=0, min=0, max=170, description='Gas:')\n",
    "    stop_btn = widgets.Button(description=\"STOP\", button_style='danger')\n",
    "\n",
    "    def on_change(change):\n",
    "        val = change['new']\n",
    "        if val > 0 and val < 120: val = 120 # Batas aman\n",
    "        ser.write(f\"{val}\\n\".encode())\n",
    "\n",
    "    def on_stop(b):\n",
    "        slider.value = 0\n",
    "        ser.write(b\"0\\n\")\n",
    "\n",
    "    slider.observe(on_change, names='value')\n",
    "    stop_btn.on_click(on_stop)\n",
    "\n",
    "    # Kickstart\n",
    "    ser.write(b\"200\\n\")\n",
    "    time.sleep(0.5)\n",
    "    ser.write(b\"0\\n\")\n",
    "\n",
    "    ani = FuncAnimation(fig, update_graph, interval=100, blit=True, cache_frame_data=False)\n",
    "    \n",
    "    display(widgets.HBox([slider, stop_btn]))\n",
    "    plt.show()\n",
    "\n",
    "else:\n",
    "    print(\"❌ Program berhenti karena Serial Error.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "46a361f2-e69d-402e-9071-1b6ba982a67e",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Pustaka dan konfigurasi siap.\n"
     ]
    },
    {
     "data": {
      "application/javascript": [
       "/* Put everything inside the global mpl namespace */\n",
       "/* global mpl */\n",
       "window.mpl = {};\n",
       "\n",
       "mpl.get_websocket_type = function () {\n",
       "    if (typeof WebSocket !== 'undefined') {\n",
       "        return WebSocket;\n",
       "    } else if (typeof MozWebSocket !== 'undefined') {\n",
       "        return MozWebSocket;\n",
       "    } else {\n",
       "        alert(\n",
       "            'Your browser does not have WebSocket support. ' +\n",
       "                'Please try Chrome, Safari or Firefox ≥ 6. ' +\n",
       "                'Firefox 4 and 5 are also supported but you ' +\n",
       "                'have to enable WebSockets in about:config.'\n",
       "        );\n",
       "    }\n",
       "};\n",
       "\n",
       "mpl.figure = function (figure_id, websocket, ondownload, parent_element) {\n",
       "    this.id = figure_id;\n",
       "\n",
       "    this.ws = websocket;\n",
       "\n",
       "    this.supports_binary = this.ws.binaryType !== undefined;\n",
       "\n",
       "    if (!this.supports_binary) {\n",
       "        var warnings = document.getElementById('mpl-warnings');\n",
       "        if (warnings) {\n",
       "            warnings.style.display = 'block';\n",
       "            warnings.textContent =\n",
       "                'This browser does not support binary websocket messages. ' +\n",
       "                'Performance may be slow.';\n",
       "        }\n",
       "    }\n",
       "\n",
       "    this.imageObj = new Image();\n",
       "\n",
       "    this.context = undefined;\n",
       "    this.message = undefined;\n",
       "    this.canvas = undefined;\n",
       "    this.rubberband_canvas = undefined;\n",
       "    this.rubberband_context = undefined;\n",
       "    this.format_dropdown = undefined;\n",
       "\n",
       "    this.image_mode = 'full';\n",
       "\n",
       "    this.root = document.createElement('div');\n",
       "    this.root.setAttribute('style', 'display: inline-block');\n",
       "    this._root_extra_style(this.root);\n",
       "\n",
       "    parent_element.appendChild(this.root);\n",
       "\n",
       "    this._init_header(this);\n",
       "    this._init_canvas(this);\n",
       "    this._init_toolbar(this);\n",
       "\n",
       "    var fig = this;\n",
       "\n",
       "    this.waiting = false;\n",
       "\n",
       "    this.ws.onopen = function () {\n",
       "        fig.send_message('supports_binary', { value: fig.supports_binary });\n",
       "        fig.send_message('send_image_mode', {});\n",
       "        if (fig.ratio !== 1) {\n",
       "            fig.send_message('set_device_pixel_ratio', {\n",
       "                device_pixel_ratio: fig.ratio,\n",
       "            });\n",
       "        }\n",
       "        fig.send_message('refresh', {});\n",
       "    };\n",
       "\n",
       "    this.imageObj.onload = function () {\n",
       "        if (fig.image_mode === 'full') {\n",
       "            // Full images could contain transparency (where diff images\n",
       "            // almost always do), so we need to clear the canvas so that\n",
       "            // there is no ghosting.\n",
       "            fig.context.clearRect(0, 0, fig.canvas.width, fig.canvas.height);\n",
       "        }\n",
       "        fig.context.drawImage(fig.imageObj, 0, 0);\n",
       "    };\n",
       "\n",
       "    this.imageObj.onunload = function () {\n",
       "        fig.ws.close();\n",
       "    };\n",
       "\n",
       "    this.ws.onmessage = this._make_on_message_function(this);\n",
       "\n",
       "    this.ondownload = ondownload;\n",
       "};\n",
       "\n",
       "mpl.figure.prototype._init_header = function () {\n",
       "    var titlebar = document.createElement('div');\n",
       "    titlebar.classList =\n",
       "        'ui-dialog-titlebar ui-widget-header ui-corner-all ui-helper-clearfix';\n",
       "    var titletext = document.createElement('div');\n",
       "    titletext.classList = 'ui-dialog-title';\n",
       "    titletext.setAttribute(\n",
       "        'style',\n",
       "        'width: 100%; text-align: center; padding: 3px;'\n",
       "    );\n",
       "    titlebar.appendChild(titletext);\n",
       "    this.root.appendChild(titlebar);\n",
       "    this.header = titletext;\n",
       "};\n",
       "\n",
       "mpl.figure.prototype._canvas_extra_style = function (_canvas_div) {};\n",
       "\n",
       "mpl.figure.prototype._root_extra_style = function (_canvas_div) {};\n",
       "\n",
       "mpl.figure.prototype._init_canvas = function () {\n",
       "    var fig = this;\n",
       "\n",
       "    var canvas_div = (this.canvas_div = document.createElement('div'));\n",
       "    canvas_div.setAttribute('tabindex', '0');\n",
       "    canvas_div.setAttribute(\n",
       "        'style',\n",
       "        'border: 1px solid #ddd;' +\n",
       "            'box-sizing: content-box;' +\n",
       "            'clear: both;' +\n",
       "            'min-height: 1px;' +\n",
       "            'min-width: 1px;' +\n",
       "            'outline: 0;' +\n",
       "            'overflow: hidden;' +\n",
       "            'position: relative;' +\n",
       "            'resize: both;' +\n",
       "            'z-index: 2;'\n",
       "    );\n",
       "\n",
       "    function on_keyboard_event_closure(name) {\n",
       "        return function (event) {\n",
       "            return fig.key_event(event, name);\n",
       "        };\n",
       "    }\n",
       "\n",
       "    canvas_div.addEventListener(\n",
       "        'keydown',\n",
       "        on_keyboard_event_closure('key_press')\n",
       "    );\n",
       "    canvas_div.addEventListener(\n",
       "        'keyup',\n",
       "        on_keyboard_event_closure('key_release')\n",
       "    );\n",
       "\n",
       "    this._canvas_extra_style(canvas_div);\n",
       "    this.root.appendChild(canvas_div);\n",
       "\n",
       "    var canvas = (this.canvas = document.createElement('canvas'));\n",
       "    canvas.classList.add('mpl-canvas');\n",
       "    canvas.setAttribute(\n",
       "        'style',\n",
       "        'box-sizing: content-box;' +\n",
       "            'pointer-events: none;' +\n",
       "            'position: relative;' +\n",
       "            'z-index: 0;'\n",
       "    );\n",
       "\n",
       "    this.context = canvas.getContext('2d');\n",
       "\n",
       "    var backingStore =\n",
       "        this.context.backingStorePixelRatio ||\n",
       "        this.context.webkitBackingStorePixelRatio ||\n",
       "        this.context.mozBackingStorePixelRatio ||\n",
       "        this.context.msBackingStorePixelRatio ||\n",
       "        this.context.oBackingStorePixelRatio ||\n",
       "        this.context.backingStorePixelRatio ||\n",
       "        1;\n",
       "\n",
       "    this.ratio = (window.devicePixelRatio || 1) / backingStore;\n",
       "\n",
       "    var rubberband_canvas = (this.rubberband_canvas = document.createElement(\n",
       "        'canvas'\n",
       "    ));\n",
       "    rubberband_canvas.setAttribute(\n",
       "        'style',\n",
       "        'box-sizing: content-box;' +\n",
       "            'left: 0;' +\n",
       "            'pointer-events: none;' +\n",
       "            'position: absolute;' +\n",
       "            'top: 0;' +\n",
       "            'z-index: 1;'\n",
       "    );\n",
       "\n",
       "    // Apply a ponyfill if ResizeObserver is not implemented by browser.\n",
       "    if (this.ResizeObserver === undefined) {\n",
       "        if (window.ResizeObserver !== undefined) {\n",
       "            this.ResizeObserver = window.ResizeObserver;\n",
       "        } else {\n",
       "            var obs = _JSXTOOLS_RESIZE_OBSERVER({});\n",
       "            this.ResizeObserver = obs.ResizeObserver;\n",
       "        }\n",
       "    }\n",
       "\n",
       "    this.resizeObserverInstance = new this.ResizeObserver(function (entries) {\n",
       "        // There's no need to resize if the WebSocket is not connected:\n",
       "        // - If it is still connecting, then we will get an initial resize from\n",
       "        //   Python once it connects.\n",
       "        // - If it has disconnected, then resizing will clear the canvas and\n",
       "        //   never get anything back to refill it, so better to not resize and\n",
       "        //   keep something visible.\n",
       "        if (fig.ws.readyState != 1) {\n",
       "            return;\n",
       "        }\n",
       "        var nentries = entries.length;\n",
       "        for (var i = 0; i < nentries; i++) {\n",
       "            var entry = entries[i];\n",
       "            var width, height;\n",
       "            if (entry.contentBoxSize) {\n",
       "                if (entry.contentBoxSize instanceof Array) {\n",
       "                    // Chrome 84 implements new version of spec.\n",
       "                    width = entry.contentBoxSize[0].inlineSize;\n",
       "                    height = entry.contentBoxSize[0].blockSize;\n",
       "                } else {\n",
       "                    // Firefox implements old version of spec.\n",
       "                    width = entry.contentBoxSize.inlineSize;\n",
       "                    height = entry.contentBoxSize.blockSize;\n",
       "                }\n",
       "            } else {\n",
       "                // Chrome <84 implements even older version of spec.\n",
       "                width = entry.contentRect.width;\n",
       "                height = entry.contentRect.height;\n",
       "            }\n",
       "\n",
       "            // Keep the size of the canvas and rubber band canvas in sync with\n",
       "            // the canvas container.\n",
       "            if (entry.devicePixelContentBoxSize) {\n",
       "                // Chrome 84 implements new version of spec.\n",
       "                canvas.setAttribute(\n",
       "                    'width',\n",
       "                    entry.devicePixelContentBoxSize[0].inlineSize\n",
       "                );\n",
       "                canvas.setAttribute(\n",
       "                    'height',\n",
       "                    entry.devicePixelContentBoxSize[0].blockSize\n",
       "                );\n",
       "            } else {\n",
       "                canvas.setAttribute('width', width * fig.ratio);\n",
       "                canvas.setAttribute('height', height * fig.ratio);\n",
       "            }\n",
       "            /* This rescales the canvas back to display pixels, so that it\n",
       "             * appears correct on HiDPI screens. */\n",
       "            canvas.style.width = width + 'px';\n",
       "            canvas.style.height = height + 'px';\n",
       "\n",
       "            rubberband_canvas.setAttribute('width', width);\n",
       "            rubberband_canvas.setAttribute('height', height);\n",
       "\n",
       "            // And update the size in Python. We ignore the initial 0/0 size\n",
       "            // that occurs as the element is placed into the DOM, which should\n",
       "            // otherwise not happen due to the minimum size styling.\n",
       "            if (width != 0 && height != 0) {\n",
       "                fig.request_resize(width, height);\n",
       "            }\n",
       "        }\n",
       "    });\n",
       "    this.resizeObserverInstance.observe(canvas_div);\n",
       "\n",
       "    function on_mouse_event_closure(name) {\n",
       "        /* User Agent sniffing is bad, but WebKit is busted:\n",
       "         * https://bugs.webkit.org/show_bug.cgi?id=144526\n",
       "         * https://bugs.webkit.org/show_bug.cgi?id=181818\n",
       "         * The worst that happens here is that they get an extra browser\n",
       "         * selection when dragging, if this check fails to catch them.\n",
       "         */\n",
       "        var UA = navigator.userAgent;\n",
       "        var isWebKit = /AppleWebKit/.test(UA) && !/Chrome/.test(UA);\n",
       "        if(isWebKit) {\n",
       "            return function (event) {\n",
       "                /* This prevents the web browser from automatically changing to\n",
       "                 * the text insertion cursor when the button is pressed. We\n",
       "                 * want to control all of the cursor setting manually through\n",
       "                 * the 'cursor' event from matplotlib */\n",
       "                event.preventDefault()\n",
       "                return fig.mouse_event(event, name);\n",
       "            };\n",
       "        } else {\n",
       "            return function (event) {\n",
       "                return fig.mouse_event(event, name);\n",
       "            };\n",
       "        }\n",
       "    }\n",
       "\n",
       "    canvas_div.addEventListener(\n",
       "        'mousedown',\n",
       "        on_mouse_event_closure('button_press')\n",
       "    );\n",
       "    canvas_div.addEventListener(\n",
       "        'mouseup',\n",
       "        on_mouse_event_closure('button_release')\n",
       "    );\n",
       "    canvas_div.addEventListener(\n",
       "        'dblclick',\n",
       "        on_mouse_event_closure('dblclick')\n",
       "    );\n",
       "    // Throttle sequential mouse events to 1 every 20ms.\n",
       "    canvas_div.addEventListener(\n",
       "        'mousemove',\n",
       "        on_mouse_event_closure('motion_notify')\n",
       "    );\n",
       "\n",
       "    canvas_div.addEventListener(\n",
       "        'mouseenter',\n",
       "        on_mouse_event_closure('figure_enter')\n",
       "    );\n",
       "    canvas_div.addEventListener(\n",
       "        'mouseleave',\n",
       "        on_mouse_event_closure('figure_leave')\n",
       "    );\n",
       "\n",
       "    canvas_div.addEventListener('wheel', function (event) {\n",
       "        if (event.deltaY < 0) {\n",
       "            event.step = 1;\n",
       "        } else {\n",
       "            event.step = -1;\n",
       "        }\n",
       "        on_mouse_event_closure('scroll')(event);\n",
       "    });\n",
       "\n",
       "    canvas_div.appendChild(canvas);\n",
       "    canvas_div.appendChild(rubberband_canvas);\n",
       "\n",
       "    this.rubberband_context = rubberband_canvas.getContext('2d');\n",
       "    this.rubberband_context.strokeStyle = '#000000';\n",
       "\n",
       "    this._resize_canvas = function (width, height, forward) {\n",
       "        if (forward) {\n",
       "            canvas_div.style.width = width + 'px';\n",
       "            canvas_div.style.height = height + 'px';\n",
       "        }\n",
       "    };\n",
       "\n",
       "    // Disable right mouse context menu.\n",
       "    canvas_div.addEventListener('contextmenu', function (_e) {\n",
       "        event.preventDefault();\n",
       "        return false;\n",
       "    });\n",
       "\n",
       "    function set_focus() {\n",
       "        canvas.focus();\n",
       "        canvas_div.focus();\n",
       "    }\n",
       "\n",
       "    window.setTimeout(set_focus, 100);\n",
       "};\n",
       "\n",
       "mpl.figure.prototype._init_toolbar = function () {\n",
       "    var fig = this;\n",
       "\n",
       "    var toolbar = document.createElement('div');\n",
       "    toolbar.classList = 'mpl-toolbar';\n",
       "    this.root.appendChild(toolbar);\n",
       "\n",
       "    function on_click_closure(name) {\n",
       "        return function (_event) {\n",
       "            return fig.toolbar_button_onclick(name);\n",
       "        };\n",
       "    }\n",
       "\n",
       "    function on_mouseover_closure(tooltip) {\n",
       "        return function (event) {\n",
       "            if (!event.currentTarget.disabled) {\n",
       "                return fig.toolbar_button_onmouseover(tooltip);\n",
       "            }\n",
       "        };\n",
       "    }\n",
       "\n",
       "    fig.buttons = {};\n",
       "    var buttonGroup = document.createElement('div');\n",
       "    buttonGroup.classList = 'mpl-button-group';\n",
       "    for (var toolbar_ind in mpl.toolbar_items) {\n",
       "        var name = mpl.toolbar_items[toolbar_ind][0];\n",
       "        var tooltip = mpl.toolbar_items[toolbar_ind][1];\n",
       "        var image = mpl.toolbar_items[toolbar_ind][2];\n",
       "        var method_name = mpl.toolbar_items[toolbar_ind][3];\n",
       "\n",
       "        if (!name) {\n",
       "            /* Instead of a spacer, we start a new button group. */\n",
       "            if (buttonGroup.hasChildNodes()) {\n",
       "                toolbar.appendChild(buttonGroup);\n",
       "            }\n",
       "            buttonGroup = document.createElement('div');\n",
       "            buttonGroup.classList = 'mpl-button-group';\n",
       "            continue;\n",
       "        }\n",
       "\n",
       "        var button = (fig.buttons[name] = document.createElement('button'));\n",
       "        button.classList = 'mpl-widget';\n",
       "        button.setAttribute('role', 'button');\n",
       "        button.setAttribute('aria-disabled', 'false');\n",
       "        button.addEventListener('click', on_click_closure(method_name));\n",
       "        button.addEventListener('mouseover', on_mouseover_closure(tooltip));\n",
       "\n",
       "        var icon_img = document.createElement('img');\n",
       "        icon_img.src = '_images/' + image + '.png';\n",
       "        icon_img.srcset = '_images/' + image + '_large.png 2x';\n",
       "        icon_img.alt = tooltip;\n",
       "        button.appendChild(icon_img);\n",
       "\n",
       "        buttonGroup.appendChild(button);\n",
       "    }\n",
       "\n",
       "    if (buttonGroup.hasChildNodes()) {\n",
       "        toolbar.appendChild(buttonGroup);\n",
       "    }\n",
       "\n",
       "    var fmt_picker = document.createElement('select');\n",
       "    fmt_picker.classList = 'mpl-widget';\n",
       "    toolbar.appendChild(fmt_picker);\n",
       "    this.format_dropdown = fmt_picker;\n",
       "\n",
       "    for (var ind in mpl.extensions) {\n",
       "        var fmt = mpl.extensions[ind];\n",
       "        var option = document.createElement('option');\n",
       "        option.selected = fmt === mpl.default_extension;\n",
       "        option.innerHTML = fmt;\n",
       "        fmt_picker.appendChild(option);\n",
       "    }\n",
       "\n",
       "    var status_bar = document.createElement('span');\n",
       "    status_bar.classList = 'mpl-message';\n",
       "    toolbar.appendChild(status_bar);\n",
       "    this.message = status_bar;\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.request_resize = function (x_pixels, y_pixels) {\n",
       "    // Request matplotlib to resize the figure. Matplotlib will then trigger a resize in the client,\n",
       "    // which will in turn request a refresh of the image.\n",
       "    this.send_message('resize', { width: x_pixels, height: y_pixels });\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.send_message = function (type, properties) {\n",
       "    properties['type'] = type;\n",
       "    properties['figure_id'] = this.id;\n",
       "    this.ws.send(JSON.stringify(properties));\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.send_draw_message = function () {\n",
       "    if (!this.waiting) {\n",
       "        this.waiting = true;\n",
       "        this.ws.send(JSON.stringify({ type: 'draw', figure_id: this.id }));\n",
       "    }\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_save = function (fig, _msg) {\n",
       "    var format_dropdown = fig.format_dropdown;\n",
       "    var format = format_dropdown.options[format_dropdown.selectedIndex].value;\n",
       "    fig.ondownload(fig, format);\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_resize = function (fig, msg) {\n",
       "    var size = msg['size'];\n",
       "    if (size[0] !== fig.canvas.width || size[1] !== fig.canvas.height) {\n",
       "        fig._resize_canvas(size[0], size[1], msg['forward']);\n",
       "        fig.send_message('refresh', {});\n",
       "    }\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_rubberband = function (fig, msg) {\n",
       "    var x0 = msg['x0'] / fig.ratio;\n",
       "    var y0 = (fig.canvas.height - msg['y0']) / fig.ratio;\n",
       "    var x1 = msg['x1'] / fig.ratio;\n",
       "    var y1 = (fig.canvas.height - msg['y1']) / fig.ratio;\n",
       "    x0 = Math.floor(x0) + 0.5;\n",
       "    y0 = Math.floor(y0) + 0.5;\n",
       "    x1 = Math.floor(x1) + 0.5;\n",
       "    y1 = Math.floor(y1) + 0.5;\n",
       "    var min_x = Math.min(x0, x1);\n",
       "    var min_y = Math.min(y0, y1);\n",
       "    var width = Math.abs(x1 - x0);\n",
       "    var height = Math.abs(y1 - y0);\n",
       "\n",
       "    fig.rubberband_context.clearRect(\n",
       "        0,\n",
       "        0,\n",
       "        fig.canvas.width / fig.ratio,\n",
       "        fig.canvas.height / fig.ratio\n",
       "    );\n",
       "\n",
       "    fig.rubberband_context.strokeRect(min_x, min_y, width, height);\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_figure_label = function (fig, msg) {\n",
       "    // Updates the figure title.\n",
       "    fig.header.textContent = msg['label'];\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_cursor = function (fig, msg) {\n",
       "    fig.canvas_div.style.cursor = msg['cursor'];\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_message = function (fig, msg) {\n",
       "    fig.message.textContent = msg['message'];\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_draw = function (fig, _msg) {\n",
       "    // Request the server to send over a new figure.\n",
       "    fig.send_draw_message();\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_image_mode = function (fig, msg) {\n",
       "    fig.image_mode = msg['mode'];\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_history_buttons = function (fig, msg) {\n",
       "    for (var key in msg) {\n",
       "        if (!(key in fig.buttons)) {\n",
       "            continue;\n",
       "        }\n",
       "        fig.buttons[key].disabled = !msg[key];\n",
       "        fig.buttons[key].setAttribute('aria-disabled', !msg[key]);\n",
       "    }\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_navigate_mode = function (fig, msg) {\n",
       "    if (msg['mode'] === 'PAN') {\n",
       "        fig.buttons['Pan'].classList.add('active');\n",
       "        fig.buttons['Zoom'].classList.remove('active');\n",
       "    } else if (msg['mode'] === 'ZOOM') {\n",
       "        fig.buttons['Pan'].classList.remove('active');\n",
       "        fig.buttons['Zoom'].classList.add('active');\n",
       "    } else {\n",
       "        fig.buttons['Pan'].classList.remove('active');\n",
       "        fig.buttons['Zoom'].classList.remove('active');\n",
       "    }\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.updated_canvas_event = function () {\n",
       "    // Called whenever the canvas gets updated.\n",
       "    this.send_message('ack', {});\n",
       "};\n",
       "\n",
       "// A function to construct a web socket function for onmessage handling.\n",
       "// Called in the figure constructor.\n",
       "mpl.figure.prototype._make_on_message_function = function (fig) {\n",
       "    return function socket_on_message(evt) {\n",
       "        if (evt.data instanceof Blob) {\n",
       "            var img = evt.data;\n",
       "            if (img.type !== 'image/png') {\n",
       "                /* FIXME: We get \"Resource interpreted as Image but\n",
       "                 * transferred with MIME type text/plain:\" errors on\n",
       "                 * Chrome.  But how to set the MIME type?  It doesn't seem\n",
       "                 * to be part of the websocket stream */\n",
       "                img.type = 'image/png';\n",
       "            }\n",
       "\n",
       "            /* Free the memory for the previous frames */\n",
       "            if (fig.imageObj.src) {\n",
       "                (window.URL || window.webkitURL).revokeObjectURL(\n",
       "                    fig.imageObj.src\n",
       "                );\n",
       "            }\n",
       "\n",
       "            fig.imageObj.src = (window.URL || window.webkitURL).createObjectURL(\n",
       "                img\n",
       "            );\n",
       "            fig.updated_canvas_event();\n",
       "            fig.waiting = false;\n",
       "            return;\n",
       "        } else if (\n",
       "            typeof evt.data === 'string' &&\n",
       "            evt.data.slice(0, 21) === 'data:image/png;base64'\n",
       "        ) {\n",
       "            fig.imageObj.src = evt.data;\n",
       "            fig.updated_canvas_event();\n",
       "            fig.waiting = false;\n",
       "            return;\n",
       "        }\n",
       "\n",
       "        var msg = JSON.parse(evt.data);\n",
       "        var msg_type = msg['type'];\n",
       "\n",
       "        // Call the  \"handle_{type}\" callback, which takes\n",
       "        // the figure and JSON message as its only arguments.\n",
       "        try {\n",
       "            var callback = fig['handle_' + msg_type];\n",
       "        } catch (e) {\n",
       "            console.log(\n",
       "                \"No handler for the '%s' message type: \",\n",
       "                msg_type,\n",
       "                msg\n",
       "            );\n",
       "            return;\n",
       "        }\n",
       "\n",
       "        if (callback) {\n",
       "            try {\n",
       "                // console.log(\"Handling '%s' message: \", msg_type, msg);\n",
       "                callback(fig, msg);\n",
       "            } catch (e) {\n",
       "                console.log(\n",
       "                    \"Exception inside the 'handler_%s' callback:\",\n",
       "                    msg_type,\n",
       "                    e,\n",
       "                    e.stack,\n",
       "                    msg\n",
       "                );\n",
       "            }\n",
       "        }\n",
       "    };\n",
       "};\n",
       "\n",
       "function getModifiers(event) {\n",
       "    var mods = [];\n",
       "    if (event.ctrlKey) {\n",
       "        mods.push('ctrl');\n",
       "    }\n",
       "    if (event.altKey) {\n",
       "        mods.push('alt');\n",
       "    }\n",
       "    if (event.shiftKey) {\n",
       "        mods.push('shift');\n",
       "    }\n",
       "    if (event.metaKey) {\n",
       "        mods.push('meta');\n",
       "    }\n",
       "    return mods;\n",
       "}\n",
       "\n",
       "/*\n",
       " * return a copy of an object with only non-object keys\n",
       " * we need this to avoid circular references\n",
       " * https://stackoverflow.com/a/24161582/3208463\n",
       " */\n",
       "function simpleKeys(original) {\n",
       "    return Object.keys(original).reduce(function (obj, key) {\n",
       "        if (typeof original[key] !== 'object') {\n",
       "            obj[key] = original[key];\n",
       "        }\n",
       "        return obj;\n",
       "    }, {});\n",
       "}\n",
       "\n",
       "mpl.figure.prototype.mouse_event = function (event, name) {\n",
       "    if (name === 'button_press') {\n",
       "        this.canvas.focus();\n",
       "        this.canvas_div.focus();\n",
       "    }\n",
       "\n",
       "    // from https://stackoverflow.com/q/1114465\n",
       "    var boundingRect = this.canvas.getBoundingClientRect();\n",
       "    var x = (event.clientX - boundingRect.left) * this.ratio;\n",
       "    var y = (event.clientY - boundingRect.top) * this.ratio;\n",
       "\n",
       "    this.send_message(name, {\n",
       "        x: x,\n",
       "        y: y,\n",
       "        button: event.button,\n",
       "        step: event.step,\n",
       "        buttons: event.buttons,\n",
       "        modifiers: getModifiers(event),\n",
       "        guiEvent: simpleKeys(event),\n",
       "    });\n",
       "\n",
       "    return false;\n",
       "};\n",
       "\n",
       "mpl.figure.prototype._key_event_extra = function (_event, _name) {\n",
       "    // Handle any extra behaviour associated with a key event\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.key_event = function (event, name) {\n",
       "    // Prevent repeat events\n",
       "    if (name === 'key_press') {\n",
       "        if (event.key === this._key) {\n",
       "            return;\n",
       "        } else {\n",
       "            this._key = event.key;\n",
       "        }\n",
       "    }\n",
       "    if (name === 'key_release') {\n",
       "        this._key = null;\n",
       "    }\n",
       "\n",
       "    var value = '';\n",
       "    if (event.ctrlKey && event.key !== 'Control') {\n",
       "        value += 'ctrl+';\n",
       "    }\n",
       "    else if (event.altKey && event.key !== 'Alt') {\n",
       "        value += 'alt+';\n",
       "    }\n",
       "    else if (event.shiftKey && event.key !== 'Shift') {\n",
       "        value += 'shift+';\n",
       "    }\n",
       "\n",
       "    value += 'k' + event.key;\n",
       "\n",
       "    this._key_event_extra(event, name);\n",
       "\n",
       "    this.send_message(name, { key: value, guiEvent: simpleKeys(event) });\n",
       "    return false;\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.toolbar_button_onclick = function (name) {\n",
       "    if (name === 'download') {\n",
       "        this.handle_save(this, null);\n",
       "    } else {\n",
       "        this.send_message('toolbar_button', { name: name });\n",
       "    }\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.toolbar_button_onmouseover = function (tooltip) {\n",
       "    this.message.textContent = tooltip;\n",
       "};\n",
       "\n",
       "///////////////// REMAINING CONTENT GENERATED BY embed_js.py /////////////////\n",
       "// prettier-ignore\n",
       "var _JSXTOOLS_RESIZE_OBSERVER=function(A){var t,i=new WeakMap,n=new WeakMap,a=new WeakMap,r=new WeakMap,o=new Set;function s(e){if(!(this instanceof s))throw new TypeError(\"Constructor requires 'new' operator\");i.set(this,e)}function h(){throw new TypeError(\"Function is not a constructor\")}function c(e,t,i,n){e=0 in arguments?Number(arguments[0]):0,t=1 in arguments?Number(arguments[1]):0,i=2 in arguments?Number(arguments[2]):0,n=3 in arguments?Number(arguments[3]):0,this.right=(this.x=this.left=e)+(this.width=i),this.bottom=(this.y=this.top=t)+(this.height=n),Object.freeze(this)}function d(){t=requestAnimationFrame(d);var s=new WeakMap,p=new Set;o.forEach((function(t){r.get(t).forEach((function(i){var r=t instanceof window.SVGElement,o=a.get(t),d=r?0:parseFloat(o.paddingTop),f=r?0:parseFloat(o.paddingRight),l=r?0:parseFloat(o.paddingBottom),u=r?0:parseFloat(o.paddingLeft),g=r?0:parseFloat(o.borderTopWidth),m=r?0:parseFloat(o.borderRightWidth),w=r?0:parseFloat(o.borderBottomWidth),b=u+f,F=d+l,v=(r?0:parseFloat(o.borderLeftWidth))+m,W=g+w,y=r?0:t.offsetHeight-W-t.clientHeight,E=r?0:t.offsetWidth-v-t.clientWidth,R=b+v,z=F+W,M=r?t.width:parseFloat(o.width)-R-E,O=r?t.height:parseFloat(o.height)-z-y;if(n.has(t)){var k=n.get(t);if(k[0]===M&&k[1]===O)return}n.set(t,[M,O]);var S=Object.create(h.prototype);S.target=t,S.contentRect=new c(u,d,M,O),s.has(i)||(s.set(i,[]),p.add(i)),s.get(i).push(S)}))})),p.forEach((function(e){i.get(e).call(e,s.get(e),e)}))}return s.prototype.observe=function(i){if(i instanceof window.Element){r.has(i)||(r.set(i,new Set),o.add(i),a.set(i,window.getComputedStyle(i)));var n=r.get(i);n.has(this)||n.add(this),cancelAnimationFrame(t),t=requestAnimationFrame(d)}},s.prototype.unobserve=function(i){if(i instanceof window.Element&&r.has(i)){var n=r.get(i);n.has(this)&&(n.delete(this),n.size||(r.delete(i),o.delete(i))),n.size||r.delete(i),o.size||cancelAnimationFrame(t)}},A.DOMRectReadOnly=c,A.ResizeObserver=s,A.ResizeObserverEntry=h,A}; // eslint-disable-line\n",
       "mpl.toolbar_items = [[\"Home\", \"Reset original view\", \"fa fa-home\", \"home\"], [\"Back\", \"Back to previous view\", \"fa fa-arrow-left\", \"back\"], [\"Forward\", \"Forward to next view\", \"fa fa-arrow-right\", \"forward\"], [\"\", \"\", \"\", \"\"], [\"Pan\", \"Left button pans, Right button zooms\\nx/y fixes axis, CTRL fixes aspect\", \"fa fa-arrows\", \"pan\"], [\"Zoom\", \"Zoom to rectangle\\nx/y fixes axis\", \"fa fa-square-o\", \"zoom\"], [\"\", \"\", \"\", \"\"], [\"Download\", \"Download plot\", \"fa fa-floppy-o\", \"download\"]];\n",
       "\n",
       "mpl.extensions = [\"eps\", \"jpeg\", \"pgf\", \"pdf\", \"png\", \"ps\", \"raw\", \"svg\", \"tif\", \"webp\"];\n",
       "\n",
       "mpl.default_extension = \"png\";/* global mpl */\n",
       "\n",
       "var comm_websocket_adapter = function (comm) {\n",
       "    // Create a \"websocket\"-like object which calls the given IPython comm\n",
       "    // object with the appropriate methods. Currently this is a non binary\n",
       "    // socket, so there is still some room for performance tuning.\n",
       "    var ws = {};\n",
       "\n",
       "    ws.binaryType = comm.kernel.ws.binaryType;\n",
       "    ws.readyState = comm.kernel.ws.readyState;\n",
       "    function updateReadyState(_event) {\n",
       "        if (comm.kernel.ws) {\n",
       "            ws.readyState = comm.kernel.ws.readyState;\n",
       "        } else {\n",
       "            ws.readyState = 3; // Closed state.\n",
       "        }\n",
       "    }\n",
       "    comm.kernel.ws.addEventListener('open', updateReadyState);\n",
       "    comm.kernel.ws.addEventListener('close', updateReadyState);\n",
       "    comm.kernel.ws.addEventListener('error', updateReadyState);\n",
       "\n",
       "    ws.close = function () {\n",
       "        comm.close();\n",
       "    };\n",
       "    ws.send = function (m) {\n",
       "        //console.log('sending', m);\n",
       "        comm.send(m);\n",
       "    };\n",
       "    // Register the callback with on_msg.\n",
       "    comm.on_msg(function (msg) {\n",
       "        //console.log('receiving', msg['content']['data'], msg);\n",
       "        var data = msg['content']['data'];\n",
       "        if (data['blob'] !== undefined) {\n",
       "            data = {\n",
       "                data: new Blob(msg['buffers'], { type: data['blob'] }),\n",
       "            };\n",
       "        }\n",
       "        // Pass the mpl event to the overridden (by mpl) onmessage function.\n",
       "        ws.onmessage(data);\n",
       "    });\n",
       "    return ws;\n",
       "};\n",
       "\n",
       "mpl.mpl_figure_comm = function (comm, msg) {\n",
       "    // This is the function which gets called when the mpl process\n",
       "    // starts-up an IPython Comm through the \"matplotlib\" channel.\n",
       "\n",
       "    var id = msg.content.data.id;\n",
       "    // Get hold of the div created by the display call when the Comm\n",
       "    // socket was opened in Python.\n",
       "    var element = document.getElementById(id);\n",
       "    var ws_proxy = comm_websocket_adapter(comm);\n",
       "\n",
       "    function ondownload(figure, _format) {\n",
       "        window.open(figure.canvas.toDataURL());\n",
       "    }\n",
       "\n",
       "    var fig = new mpl.figure(id, ws_proxy, ondownload, element);\n",
       "\n",
       "    // Call onopen now - mpl needs it, as it is assuming we've passed it a real\n",
       "    // web socket which is closed, not our websocket->open comm proxy.\n",
       "    ws_proxy.onopen();\n",
       "\n",
       "    fig.parent_element = element;\n",
       "    fig.cell_info = mpl.find_output_cell(\"<div id='\" + id + \"'></div>\");\n",
       "    if (!fig.cell_info) {\n",
       "        console.error('Failed to find cell for figure', id, fig);\n",
       "        return;\n",
       "    }\n",
       "    fig.cell_info[0].output_area.element.on(\n",
       "        'cleared',\n",
       "        { fig: fig },\n",
       "        fig._remove_fig_handler\n",
       "    );\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_close = function (fig, msg) {\n",
       "    var width = fig.canvas.width / fig.ratio;\n",
       "    fig.cell_info[0].output_area.element.off(\n",
       "        'cleared',\n",
       "        fig._remove_fig_handler\n",
       "    );\n",
       "    fig.resizeObserverInstance.unobserve(fig.canvas_div);\n",
       "\n",
       "    // Update the output cell to use the data from the current canvas.\n",
       "    fig.push_to_output();\n",
       "    var dataURL = fig.canvas.toDataURL();\n",
       "    // Re-enable the keyboard manager in IPython - without this line, in FF,\n",
       "    // the notebook keyboard shortcuts fail.\n",
       "    IPython.keyboard_manager.enable();\n",
       "    fig.parent_element.innerHTML =\n",
       "        '<img src=\"' + dataURL + '\" width=\"' + width + '\">';\n",
       "    fig.close_ws(fig, msg);\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.close_ws = function (fig, msg) {\n",
       "    fig.send_message('closing', msg);\n",
       "    // fig.ws.close()\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.push_to_output = function (_remove_interactive) {\n",
       "    // Turn the data on the canvas into data in the output cell.\n",
       "    var width = this.canvas.width / this.ratio;\n",
       "    var dataURL = this.canvas.toDataURL();\n",
       "    this.cell_info[1]['text/html'] =\n",
       "        '<img src=\"' + dataURL + '\" width=\"' + width + '\">';\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.updated_canvas_event = function () {\n",
       "    // Tell IPython that the notebook contents must change.\n",
       "    IPython.notebook.set_dirty(true);\n",
       "    this.send_message('ack', {});\n",
       "    var fig = this;\n",
       "    // Wait a second, then push the new image to the DOM so\n",
       "    // that it is saved nicely (might be nice to debounce this).\n",
       "    setTimeout(function () {\n",
       "        fig.push_to_output();\n",
       "    }, 1000);\n",
       "};\n",
       "\n",
       "mpl.figure.prototype._init_toolbar = function () {\n",
       "    var fig = this;\n",
       "\n",
       "    var toolbar = document.createElement('div');\n",
       "    toolbar.classList = 'btn-toolbar';\n",
       "    this.root.appendChild(toolbar);\n",
       "\n",
       "    function on_click_closure(name) {\n",
       "        return function (_event) {\n",
       "            return fig.toolbar_button_onclick(name);\n",
       "        };\n",
       "    }\n",
       "\n",
       "    function on_mouseover_closure(tooltip) {\n",
       "        return function (event) {\n",
       "            if (!event.currentTarget.disabled) {\n",
       "                return fig.toolbar_button_onmouseover(tooltip);\n",
       "            }\n",
       "        };\n",
       "    }\n",
       "\n",
       "    fig.buttons = {};\n",
       "    var buttonGroup = document.createElement('div');\n",
       "    buttonGroup.classList = 'btn-group';\n",
       "    var button;\n",
       "    for (var toolbar_ind in mpl.toolbar_items) {\n",
       "        var name = mpl.toolbar_items[toolbar_ind][0];\n",
       "        var tooltip = mpl.toolbar_items[toolbar_ind][1];\n",
       "        var image = mpl.toolbar_items[toolbar_ind][2];\n",
       "        var method_name = mpl.toolbar_items[toolbar_ind][3];\n",
       "\n",
       "        if (!name) {\n",
       "            /* Instead of a spacer, we start a new button group. */\n",
       "            if (buttonGroup.hasChildNodes()) {\n",
       "                toolbar.appendChild(buttonGroup);\n",
       "            }\n",
       "            buttonGroup = document.createElement('div');\n",
       "            buttonGroup.classList = 'btn-group';\n",
       "            continue;\n",
       "        }\n",
       "\n",
       "        button = fig.buttons[name] = document.createElement('button');\n",
       "        button.classList = 'btn btn-default';\n",
       "        button.href = '#';\n",
       "        button.title = name;\n",
       "        button.innerHTML = '<i class=\"fa ' + image + ' fa-lg\"></i>';\n",
       "        button.addEventListener('click', on_click_closure(method_name));\n",
       "        button.addEventListener('mouseover', on_mouseover_closure(tooltip));\n",
       "        buttonGroup.appendChild(button);\n",
       "    }\n",
       "\n",
       "    if (buttonGroup.hasChildNodes()) {\n",
       "        toolbar.appendChild(buttonGroup);\n",
       "    }\n",
       "\n",
       "    // Add the status bar.\n",
       "    var status_bar = document.createElement('span');\n",
       "    status_bar.classList = 'mpl-message pull-right';\n",
       "    toolbar.appendChild(status_bar);\n",
       "    this.message = status_bar;\n",
       "\n",
       "    // Add the close button to the window.\n",
       "    var buttongrp = document.createElement('div');\n",
       "    buttongrp.classList = 'btn-group inline pull-right';\n",
       "    button = document.createElement('button');\n",
       "    button.classList = 'btn btn-mini btn-primary';\n",
       "    button.href = '#';\n",
       "    button.title = 'Stop Interaction';\n",
       "    button.innerHTML = '<i class=\"fa fa-power-off icon-remove icon-large\"></i>';\n",
       "    button.addEventListener('click', function (_evt) {\n",
       "        fig.handle_close(fig, {});\n",
       "    });\n",
       "    button.addEventListener(\n",
       "        'mouseover',\n",
       "        on_mouseover_closure('Stop Interaction')\n",
       "    );\n",
       "    buttongrp.appendChild(button);\n",
       "    var titlebar = this.root.querySelector('.ui-dialog-titlebar');\n",
       "    titlebar.insertBefore(buttongrp, titlebar.firstChild);\n",
       "};\n",
       "\n",
       "mpl.figure.prototype._remove_fig_handler = function (event) {\n",
       "    var fig = event.data.fig;\n",
       "    if (event.target !== this) {\n",
       "        // Ignore bubbled events from children.\n",
       "        return;\n",
       "    }\n",
       "    fig.close_ws(fig, {});\n",
       "};\n",
       "\n",
       "mpl.figure.prototype._root_extra_style = function (el) {\n",
       "    el.style.boxSizing = 'content-box'; // override notebook setting of border-box.\n",
       "};\n",
       "\n",
       "mpl.figure.prototype._canvas_extra_style = function (el) {\n",
       "    // this is important to make the div 'focusable\n",
       "    el.setAttribute('tabindex', 0);\n",
       "    // reach out to IPython and tell the keyboard manager to turn it's self\n",
       "    // off when our div gets focus\n",
       "\n",
       "    // location in version 3\n",
       "    if (IPython.notebook.keyboard_manager) {\n",
       "        IPython.notebook.keyboard_manager.register_events(el);\n",
       "    } else {\n",
       "        // location in version 2\n",
       "        IPython.keyboard_manager.register_events(el);\n",
       "    }\n",
       "};\n",
       "\n",
       "mpl.figure.prototype._key_event_extra = function (event, _name) {\n",
       "    // Check for shift+enter\n",
       "    if (event.shiftKey && event.which === 13) {\n",
       "        this.canvas_div.blur();\n",
       "        // select the cell after this one\n",
       "        var index = IPython.notebook.find_cell_index(this.cell_info[0]);\n",
       "        IPython.notebook.select(index + 1);\n",
       "    }\n",
       "};\n",
       "\n",
       "mpl.figure.prototype.handle_save = function (fig, _msg) {\n",
       "    fig.ondownload(fig, null);\n",
       "};\n",
       "\n",
       "mpl.find_output_cell = function (html_output) {\n",
       "    // Return the cell and output element which can be found *uniquely* in the notebook.\n",
       "    // Note - this is a bit hacky, but it is done because the \"notebook_saving.Notebook\"\n",
       "    // IPython event is triggered only after the cells have been serialised, which for\n",
       "    // our purposes (turning an active figure into a static one), is too late.\n",
       "    var cells = IPython.notebook.get_cells();\n",
       "    var ncells = cells.length;\n",
       "    for (var i = 0; i < ncells; i++) {\n",
       "        var cell = cells[i];\n",
       "        if (cell.cell_type === 'code') {\n",
       "            for (var j = 0; j < cell.output_area.outputs.length; j++) {\n",
       "                var data = cell.output_area.outputs[j];\n",
       "                if (data.data) {\n",
       "                    // IPython >= 3 moved mimebundle to data attribute of output\n",
       "                    data = data.data;\n",
       "                }\n",
       "                if (data['text/html'] === html_output) {\n",
       "                    return [cell, data, j];\n",
       "                }\n",
       "            }\n",
       "        }\n",
       "    }\n",
       "};\n",
       "\n",
       "// Register the function which deals with the matplotlib target/channel.\n",
       "// The kernel may be null if the page has been refreshed.\n",
       "if (IPython.notebook.kernel !== null) {\n",
       "    IPython.notebook.kernel.comm_manager.register_target(\n",
       "        'matplotlib',\n",
       "        mpl.mpl_figure_comm\n",
       "    );\n",
       "}\n"
      ],
      "text/plain": [
       "<IPython.core.display.Javascript object>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<div id='56bb2142-ef46-4f2f-97e5-b038ee2a7031'></div>"
      ],
      "text/plain": [
       "<IPython.core.display.HTML object>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Slider kontrol PWM siap. Geser untuk mengontrol motor.\n"
     ]
    },
    {
     "data": {
      "application/vnd.jupyter.widget-view+json": {
       "model_id": "ee05ff87e7a14decb869427a92eaa0cc",
       "version_major": 2,
       "version_minor": 0
      },
      "text/plain": [
       "IntSlider(value=0, continuous_update=False, description='Kontrol PWM Motor:', layout=Layout(width='60%'), max=…"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "✅ Berhasil terhubung ke COM9.\n",
      "Memulai pembacaan dan visualisasi... (Untuk berhenti, klik 'Kernel' -> 'Interrupt')\n",
      "\n",
      "Koneksi serial ditutup dengan aman.\n"
     ]
    },
    {
     "ename": "KeyboardInterrupt",
     "evalue": "",
     "output_type": "error",
     "traceback": [
      "\u001b[31m---------------------------------------------------------------------------\u001b[39m",
      "\u001b[31mKeyboardInterrupt\u001b[39m                         Traceback (most recent call last)",
      "\u001b[36mCell\u001b[39m\u001b[36m \u001b[39m\u001b[32mIn[2]\u001b[39m\u001b[32m, line 134\u001b[39m\n\u001b[32m    131\u001b[39m         ax.set_ylim(bottom=-\u001b[32m10\u001b[39m, top=\u001b[38;5;28mmax\u001b[39m(max_rpm, \u001b[32m260\u001b[39m) * \u001b[32m1.1\u001b[39m)\n\u001b[32m    133\u001b[39m         fig.canvas.draw()\n\u001b[32m--> \u001b[39m\u001b[32m134\u001b[39m         \u001b[43mplt\u001b[49m\u001b[43m.\u001b[49m\u001b[43mpause\u001b[49m\u001b[43m(\u001b[49m\u001b[32;43m0.5\u001b[39;49m\u001b[43m)\u001b[49m \u001b[38;5;66;03m# Update grafik setiap 0.5 detik\u001b[39;00m\n\u001b[32m    136\u001b[39m \u001b[38;5;28;01mexcept\u001b[39;00m \u001b[38;5;167;01mException\u001b[39;00m \u001b[38;5;28;01mas\u001b[39;00m e:\n\u001b[32m    137\u001b[39m     \u001b[38;5;28mprint\u001b[39m(\u001b[33mf\u001b[39m\u001b[33m\"\u001b[39m\u001b[33m❌ Terjadi Error: \u001b[39m\u001b[38;5;132;01m{\u001b[39;00me\u001b[38;5;132;01m}\u001b[39;00m\u001b[33m\"\u001b[39m)\n",
      "\u001b[36mFile \u001b[39m\u001b[32m~\\lstm_env\\Lib\\site-packages\\matplotlib\\pyplot.py:757\u001b[39m, in \u001b[36mpause\u001b[39m\u001b[34m(interval)\u001b[39m\n\u001b[32m    755\u001b[39m         canvas.draw_idle()\n\u001b[32m    756\u001b[39m     show(block=\u001b[38;5;28;01mFalse\u001b[39;00m)\n\u001b[32m--> \u001b[39m\u001b[32m757\u001b[39m     \u001b[43mcanvas\u001b[49m\u001b[43m.\u001b[49m\u001b[43mstart_event_loop\u001b[49m\u001b[43m(\u001b[49m\u001b[43minterval\u001b[49m\u001b[43m)\u001b[49m\n\u001b[32m    758\u001b[39m \u001b[38;5;28;01melse\u001b[39;00m:\n\u001b[32m    759\u001b[39m     time.sleep(interval)\n",
      "\u001b[36mFile \u001b[39m\u001b[32m~\\lstm_env\\Lib\\site-packages\\matplotlib\\backend_bases.py:2367\u001b[39m, in \u001b[36mFigureCanvasBase.start_event_loop\u001b[39m\u001b[34m(self, timeout)\u001b[39m\n\u001b[32m   2365\u001b[39m \u001b[38;5;28;01mwhile\u001b[39;00m \u001b[38;5;28mself\u001b[39m._looping \u001b[38;5;129;01mand\u001b[39;00m counter * timestep < timeout:\n\u001b[32m   2366\u001b[39m     \u001b[38;5;28mself\u001b[39m.flush_events()\n\u001b[32m-> \u001b[39m\u001b[32m2367\u001b[39m     time.sleep(timestep)\n\u001b[32m   2368\u001b[39m     counter += \u001b[32m1\u001b[39m\n",
      "\u001b[31mKeyboardInterrupt\u001b[39m: "
     ]
    }
   ],
   "source": [
    "# ====================================================================\n",
    "# == KODE LENGKAP JUPYTER NOTEBOOK UNTUK KONTROL & VISUALISASI RPM  ==\n",
    "# ====================================================================\n",
    "\n",
    "# --------------------------------\n",
    "# Bagian 1: Impor & Konfigurasi\n",
    "# --------------------------------\n",
    "import serial\n",
    "import time\n",
    "import matplotlib.pyplot as plt\n",
    "import ipywidgets as widgets\n",
    "from IPython.display import display\n",
    "import threading\n",
    "\n",
    "# --- KONFIGURASI PENTING (UBAH JIKA PERLU) ---\n",
    "PORT = 'COM9'       # <<< GANTI DENGAN PORT ESP32 ANDA\n",
    "BAUDRATE = 115200\n",
    "PLOT_WINDOW_SIZE = 100 # Jumlah titik data yang ditampilkan di grafik\n",
    "\n",
    "# Aktifkan mode plot interaktif di Jupyter\n",
    "%matplotlib notebook\n",
    "\n",
    "print(\"Pustaka dan konfigurasi siap.\")\n",
    "\n",
    "# ----------------------------------------------------\n",
    "# Bagian 2: Inisialisasi Variabel Global & Widget\n",
    "# ----------------------------------------------------\n",
    "\n",
    "# Variabel global untuk komunikasi antar fungsi dan thread\n",
    "ser = None                # Objek koneksi serial\n",
    "stop_thread = False       # Flag untuk menghentikan thread dengan aman\n",
    "x_vals, y_rpm_vals, y_pwm_vals = [], [], [] # List untuk menyimpan data plot\n",
    "count = 0                 # Penghitung iterasi untuk sumbu X\n",
    "\n",
    "# Buat slider untuk kontrol PWM\n",
    "pwm_slider = widgets.IntSlider(\n",
    "    value=0, min=0, max=255, step=1,\n",
    "    description='Kontrol PWM Motor:',\n",
    "    continuous_update=False, # Hanya update saat slider dilepas\n",
    "    style={'description_width': 'initial'},\n",
    "    layout=widgets.Layout(width='60%')\n",
    ")\n",
    "\n",
    "# Fungsi yang dipanggil saat slider digeser\n",
    "def handle_slider_change(change):\n",
    "    pwm_value = change.new\n",
    "    if ser and ser.is_open:\n",
    "        try:\n",
    "            ser.write(f\"{pwm_value}\\n\".encode())\n",
    "            print(f\"Mengirim PWM: {pwm_value}\")\n",
    "        except Exception as e:\n",
    "            print(f\"Gagal mengirim data: {e}\")\n",
    "\n",
    "# Hubungkan fungsi di atas ke slider\n",
    "pwm_slider.observe(handle_slider_change, names='value')\n",
    "\n",
    "# ----------------------------------------------------\n",
    "# Bagian 3: Fungsi untuk Thread Pembaca Serial\n",
    "# ----------------------------------------------------\n",
    "\n",
    "def serial_reader_thread():\n",
    "    \"\"\"Fungsi ini berjalan di background untuk terus membaca data dari ESP32.\"\"\"\n",
    "    global count, x_vals, y_rpm_vals, y_pwm_vals\n",
    "    \n",
    "    while not stop_thread:\n",
    "        if ser and ser.is_open and ser.in_waiting > 0:\n",
    "            try:\n",
    "                line = ser.readline().decode('utf-8').strip()\n",
    "                if line and ',' in line:\n",
    "                    pwm_str, rpm_str = line.split(',')\n",
    "                    current_pwm = int(pwm_str.strip())\n",
    "                    current_rpm = int(rpm_str.strip())\n",
    "                    \n",
    "                    # Tambahkan data baru ke list\n",
    "                    x_vals.append(count)\n",
    "                    y_rpm_vals.append(current_rpm)\n",
    "                    y_pwm_vals.append(current_pwm)\n",
    "                    count += 1\n",
    "                    \n",
    "                    # Jaga agar list tidak terlalu besar (efek jendela geser)\n",
    "                    if len(x_vals) > PLOT_WINDOW_SIZE:\n",
    "                        x_vals.pop(0)\n",
    "                        y_rpm_vals.pop(0)\n",
    "                        y_pwm_vals.pop(0)\n",
    "\n",
    "            except (ValueError, UnicodeDecodeError):\n",
    "                continue # Abaikan data yang rusak\n",
    "        time.sleep(0.05) # Jeda singkat agar tidak membebani CPU\n",
    "\n",
    "# -----------------------------------------------------------------\n",
    "# Bagian 4: Program Utama (Menghubungkan & Memvisualisasikan)\n",
    "# -----------------------------------------------------------------\n",
    "\n",
    "# Setup grafik\n",
    "fig, ax = plt.subplots(figsize=(10, 6))\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "\n",
    "# Tampilkan slider sebelum memulai loop\n",
    "print(\"Slider kontrol PWM siap. Geser untuk mengontrol motor.\")\n",
    "display(pwm_slider)\n",
    "\n",
    "try:\n",
    "    # Buka koneksi serial\n",
    "    ser = serial.Serial(PORT, BAUDRATE, timeout=1)\n",
    "    time.sleep(2)\n",
    "    ser.reset_input_buffer()\n",
    "    print(f\"\\n✅ Berhasil terhubung ke {PORT}.\")\n",
    "\n",
    "    # Mulai thread pembaca serial\n",
    "    reader = threading.Thread(target=serial_reader_thread)\n",
    "    reader.start()\n",
    "    print(\"Memulai pembacaan dan visualisasi... (Untuk berhenti, klik 'Kernel' -> 'Interrupt')\")\n",
    "\n",
    "    # Loop utama untuk memperbarui grafik\n",
    "    while True:\n",
    "        ax.clear() # Hapus plot lama\n",
    "        \n",
    "        # Plot data RPM dan PWM\n",
    "        ax.plot(x_vals, y_rpm_vals, label='RPM Aktual (dari Sensor)', color='deepskyblue', marker='.', markersize=4)\n",
    "        ax.plot(x_vals, y_pwm_vals, label='PWM Terkirim (dari Slider)', color='tomato', linestyle='--')\n",
    "        \n",
    "        # Atur tampilan grafik\n",
    "        ax.set_title('Visualisasi & Kontrol Real-Time Kecepatan Motor', fontsize=16, pad=20)\n",
    "        ax.set_xlabel('Waktu (Sampel Data)', fontsize=12)\n",
    "        ax.set_ylabel('Nilai', fontsize=12)\n",
    "        ax.legend(loc='upper left')\n",
    "        ax.grid(True, which='both', linestyle='--', linewidth=0.5)\n",
    "        \n",
    "        # Atur batas sumbu Y secara dinamis\n",
    "        max_rpm = max(y_rpm_vals) if y_rpm_vals else 255\n",
    "        ax.set_ylim(bottom=-10, top=max(max_rpm, 260) * 1.1)\n",
    "\n",
    "        fig.canvas.draw()\n",
    "        plt.pause(0.5) # Update grafik setiap 0.5 detik\n",
    "\n",
    "except Exception as e:\n",
    "    print(f\"❌ Terjadi Error: {e}\")\n",
    "finally:\n",
    "    # Prosedur penutupan yang aman\n",
    "    stop_thread = True # Beri sinyal agar thread berhenti\n",
    "    if 'reader' in locals() and reader.is_alive():\n",
    "        reader.join() # Tunggu thread selesai\n",
    "    if ser and ser.is_open:\n",
    "        ser.write(b\"0\\n\") # Matikan motor\n",
    "        ser.close()\n",
    "        print(\"\\nKoneksi serial ditutup dengan aman.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "1c1da2f2-e470-46ef-95e1-fe34810b8aa0",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11.6"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}